### Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

### file Paths

In [2]:
def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "README.md").exists() and (path / "config.yaml").exists():
            return path

    raise FileNotFoundError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd())

RAW_DIR = PROJECT_ROOT / "data" / "raw"
YF_DIR = RAW_DIR / "yfinance"

STOCK_DIR = YF_DIR / "stocks"
INDEX_DIR = YF_DIR / "indices"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
QUALITY_REPORT_DIR = INTERIM_DIR / "quality_reports"
FEATURE_DIR = INTERIM_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

STOCK_ACCEPTANCE_PATH = QUALITY_REPORT_DIR / "stock_acceptance_report.csv"
LARGE_MOVE_REVIEW_PATH = QUALITY_REPORT_DIR / "large_daily_moves_review_report.csv"

FEATURE_OUTPUT_PARQUET = FEATURE_DIR / "stock_features_v1.parquet"
FEATURE_OUTPUT_CSV = FEATURE_DIR / "stock_features_v1.csv"
FEATURE_QUALITY_PATH = FEATURE_DIR / "feature_quality_summary_v1.csv"
FEATURE_DICTIONARY_PATH = FEATURE_DIR / "feature_dictionary_v1.csv"

print("Project root:", PROJECT_ROOT)
print("Stock dir:", STOCK_DIR)
print("Index dir:", INDEX_DIR)
print("Feature dir:", FEATURE_DIR)

Project root: E:\Projects\marketguard-india
Stock dir: E:\Projects\marketguard-india\data\raw\yfinance\stocks
Index dir: E:\Projects\marketguard-india\data\raw\yfinance\indices
Feature dir: E:\Projects\marketguard-india\data\interim\features


### Load stock and index data 

In [3]:
def read_market_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    if "date" not in df.columns:
        raise ValueError(f"No date column found in {path.name}")

    df["date"] = pd.to_datetime(df["date"])
    df["source_file"] = path.name

    return df


stock_files = sorted(STOCK_DIR.glob("*.csv"))
index_files = sorted(INDEX_DIR.glob("*.csv"))

print("Stock files:", len(stock_files))
print("Index files:", len(index_files))

stock_dfs = [read_market_file(path) for path in stock_files]
index_dfs = [read_market_file(path) for path in index_files]

stocks = pd.concat(stock_dfs, ignore_index=True)
indices = pd.concat(index_dfs, ignore_index=True)

stocks = stocks.sort_values(["yf_ticker", "date"]).reset_index(drop=True)
indices = indices.sort_values(["yf_ticker", "date"]).reset_index(drop=True)

print("Stocks shape:", stocks.shape)
print("Indices shape:", indices.shape)

display(stocks.sample(5))
display(indices.sample(5))

Stock files: 100
Index files: 16
Stocks shape: (357370, 16)
Indices shape: (61691, 16)


,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume,source_file
271322,stock,SHREECEM,SHREECEM.NS,Shree Cement Ltd.,Construction Materials,2019-11-05,19339.666016,19829.300781,0.0,20199.000000,19791.550781,20199.000000,False,0.0,17968,SHREECEM_NS.csv
217193,stock,MAZDOCK,MAZDOCK.NS,Mazagoan Dock Shipbuilders Ltd.,Capital Goods,2022-12-15,420.428406,431.924988,0.0,442.149994,430.000000,439.950012,False,0.0,2791862,MAZDOCK_NS.csv
302849,stock,TCS,TCS.NS,Tata Consultancy Services Ltd.,Information Technology,2014-06-25,868.531677,1155.275024,0.0,1160.000000,1146.500000,1155.000000,False,0.0,1028422,TCS_NS.csv
59884,stock,BEL,BEL.NS,Bharat Electronics Ltd.,Capital Goods,2018-07-31,33.425606,38.783333,0.0,39.883331,35.733334,36.133331,False,0.0,77121654,BEL_NS.csv
285434,stock,SUNPHARMA,SUNPHARMA.NS,Sun Pharmaceutical Industries Ltd.,Healthcare,2010-11-02,196.164932,219.044998,0.0,220.399994,216.615005,217.199997,False,0.0,844550,SUNPHARMA_NS.csv


,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume,source_file
52663,index,India VIX,^INDIAVIX,India VIX,Volatility,2022-11-21,14.800000,14.800000,0.0,15.550000,13.360000,14.390000,False,0.0,0,INDIAVIX.csv
24709,index,Nifty Media,^CNXMEDIA,Nifty Media,Media,2018-05-11,3303.600098,3303.600098,0.0,3317.699951,3278.649902,3282.600098,False,0.0,137500,CNXMEDIA.csv
30812,index,Nifty MNC,^CNXMNC,Nifty MNC,MNC,2012-10-12,5841.850098,5841.850098,0.0,5873.100098,5820.700195,5827.750000,False,0.0,106700,CNXMNC.csv
48135,index,Nifty Realty,^CNXREALTY,Nifty Realty,Realty,2020-12-15,289.000000,289.000000,0.0,290.049988,285.149994,289.750000,False,0.0,170800,CNXREALTY.csv
27864,index,Nifty Metal,^CNXMETAL,Nifty Metal,Metal,2016-03-28,1853.199951,1853.199951,0.0,1954.500000,1845.849976,1954.500000,False,0.0,857600,CNXMETAL.csv


### Load acceptance and large-move review reports 

In [4]:
if STOCK_ACCEPTANCE_PATH.exists():
    stock_acceptance = pd.read_csv(STOCK_ACCEPTANCE_PATH)
    print("Loaded stock acceptance:", stock_acceptance.shape)
    display(stock_acceptance["status"].value_counts())
else:
    stock_acceptance = pd.DataFrame()
    print("Stock acceptance report not found. Continuing without it.")


if LARGE_MOVE_REVIEW_PATH.exists():
    large_move_review = pd.read_csv(LARGE_MOVE_REVIEW_PATH)
    large_move_review["date"] = pd.to_datetime(large_move_review["date"])
    print("Loaded large move review:", large_move_review.shape)
    display(large_move_review["review_status"].value_counts())
else:
    large_move_review = pd.DataFrame()
    print("Large move review report not found. Continuing without it.")

Loaded stock acceptance: (100, 22)


status
OK                    90
FLAG_SHORT_HISTORY    10
Name: count, dtype: int64

Loaded large move review: (51, 28)


review_status
MANUAL_REVIEW    40
KEEP             11
Name: count, dtype: int64

### Schema check

In [5]:
print("Stock columns:")
print(stocks.columns.tolist())

print("\nIndex columns:")
print(indices.columns.tolist())

Stock columns:
['asset_type', 'symbol', 'yf_ticker', 'company_name', 'industry', 'date', 'adj_close', 'close', 'dividends', 'high', 'low', 'open', 'repaired?', 'stock_splits', 'volume', 'source_file']

Index columns:
['asset_type', 'symbol', 'yf_ticker', 'company_name', 'industry', 'date', 'adj_close', 'close', 'dividends', 'high', 'low', 'open', 'repaired?', 'stock_splits', 'volume', 'source_file']


In [6]:
required_stock_cols = [
    "date",
    "symbol",
    "yf_ticker",
    "company_name",
    "industry",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
]

required_index_cols = [
    "date",
    "symbol",
    "yf_ticker",
    "open",
    "high",
    "low",
    "close",
]

missing_stock_cols = [col for col in required_stock_cols if col not in stocks.columns]
missing_index_cols = [col for col in required_index_cols if col not in indices.columns]

print("Missing stock columns:", missing_stock_cols)
print("Missing index columns:", missing_index_cols)

print("\nStock date range:", stocks["date"].min(), "to", stocks["date"].max())
print("Index date range:", indices["date"].min(), "to", indices["date"].max())

print("\nUnique stock tickers:", stocks["yf_ticker"].nunique())
print("Unique index tickers:", indices["yf_ticker"].nunique())

Missing stock columns: []
Missing index columns: []

Stock date range: 2010-01-04 00:00:00 to 2026-07-14 00:00:00
Index date range: 2010-01-04 00:00:00 to 2026-07-14 00:00:00

Unique stock tickers: 100
Unique index tickers: 16


### Helper functions to calculate RSI

The Relative Strength Index (RSI) is a popular momentum indicator used in technical analysis to measure the magnitude and speed of recent price changes in a stock or tradable asset. Developed by J. Welles Wilder Jr., it helps traders evaluate whether an asset is potentially overvalued or undervalued

The RSI is displayed as a line graph that oscillates on a scale of 0 to 100. It is most commonly calculated using a 14-day period

->Overbought (>70): Traditionally, a reading above 70 suggests the asset may be overbought or overvalued, indicating a potential pullback or trend reversal.

->Oversold (<30): A reading below 30 indicates an oversold or undervalued condition, hinting that the asset might be due for a price bounce.

->Neutral (50): A reading near 50 represents a neutral market momentum, showing a balance between buyers and sellers

In [7]:
def safe_divide(a, b):
    #    Divide safely by replacing 0 denominator with NaN.
    return a / b.replace(0, np.nan)


def compute_rsi(price: pd.Series, period: int = 14) -> pd.Series:
     """
    Compute RSI using Wilder-style exponential smoothing.
    Input should usually be adjusted close.
    """    
     delta = price.diff()

     gain = delta.clip(lower=0)
     loss = -delta.clip(upper=0)

     avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
     avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()

     rs = avg_gain / avg_loss.replace(0, np.nan)
     rsi = 100 - (100 / (1 + rs))

     return rsi

In [8]:
sample_ticker = stocks["yf_ticker"].iloc[0]
sample = stocks[stocks["yf_ticker"] == sample_ticker].sort_values("date").copy()

sample["rsi_14_test"] = compute_rsi(sample["adj_close"], period=14)

display(
    sample[["date", "yf_ticker", "adj_close", "rsi_14_test"]]
    .dropna()
    .head()
)

,date,yf_ticker,adj_close,rsi_14_test
14,2010-01-22,ABB.NS,685.352051,48.355747
15,2010-01-25,ABB.NS,691.503418,50.620863
16,2010-01-27,ABB.NS,660.491943,40.885120
17,2010-01-28,ABB.NS,667.110046,43.387469
18,2010-01-29,ABB.NS,688.109680,50.541559


### Stock feature function

we will create base stock features such as returns,
log returns,
rolling volatility,
moving averages,
drawdown,
RSI,
history_days_available.

No future data is used here, so no target leakage.

In [9]:
def add_base_stock_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    # history_days_available:
    # Number of trading days available for this stock up to the current row.
    # Useful for filtering short-history stocks and avoiding unreliable early indicators.
    g["history_days_available"] = np.arange(1, len(g) + 1)

    # Previous close values:
    # Needed to calculate daily returns, gaps, and log returns.
    # prev_close uses raw close; prev_adj_close uses adjusted close.
    g["prev_close"] = g["close"].shift(1)
    g["prev_adj_close"] = g["adj_close"].shift(1)

    # Return features using adjusted close:
    # These measure past price performance over different lookback windows.
    # Adjusted close is used because it handles dividends/splits better than raw close.
    # Used for momentum, trend strength, and recent stock behaviour.
    g["return_1d"] = g["adj_close"].pct_change(1)
    g["return_5d"] = g["adj_close"].pct_change(5)
    g["return_20d"] = g["adj_close"].pct_change(20)
    g["return_60d"] = g["adj_close"].pct_change(60)
    g["return_120d"] = g["adj_close"].pct_change(120)

    # log_return_1d:
    # Alternative daily return format useful for volatility/statistical modelling.
    # Log returns are more stable when aggregating returns over time.
    g["log_return_1d"] = np.log(g["adj_close"] / g["prev_adj_close"])

    # raw_return_1d:
    # Daily return using raw close.
    # Mainly used for abnormal move detection and comparison against adjusted-close return.
    g["raw_return_1d"] = g["close"].pct_change(1)

    # Rolling volatility:
    # Measures how unstable the stock's recent daily returns are.
    # Higher volatility means higher risk and wider expected price movement.
    g["daily_volatility_20d"] = g["return_1d"].rolling(20).std()
    g["daily_volatility_60d"] = g["return_1d"].rolling(60).std()

    # Annualized volatility:
    # Converts daily volatility into yearly scale using sqrt(252 trading days).
    # Easier to compare across stocks and with market-level risk.
    g["annualized_volatility_20d"] = g["daily_volatility_20d"] * np.sqrt(252)
    g["annualized_volatility_60d"] = g["daily_volatility_60d"] * np.sqrt(252)

    # Moving average features:
    # Moving averages smooth price movement and show trend direction.
    # adj_close_to_ma_* tells how far price is above/below its trend level.
    # Positive value = stock trading above average; negative = below average.
    for window in [20, 50, 100, 200]:
        ma_col = f"ma_{window}"
        ratio_col = f"adj_close_to_ma_{window}"

        g[ma_col] = g["adj_close"].rolling(window).mean()
        g[ratio_col] = g["adj_close"] / g[ma_col] - 1

    # Rolling highs:
    # Highest adjusted close over recent 60 trading days and 252 trading days.
    # Used to measure drawdown from recent/local and yearly highs.
    g["rolling_high_60d"] = g["adj_close"].rolling(60).max()
    g["rolling_high_252d"] = g["adj_close"].rolling(252).max()

    # Drawdown features:
    # Measures how far the stock has fallen from its recent high.
    # More negative value means deeper fall and potentially higher risk/weakness.
    g["drawdown_from_60d_high"] = g["adj_close"] / g["rolling_high_60d"] - 1
    g["drawdown_from_252d_high"] = g["adj_close"] / g["rolling_high_252d"] - 1

    # RSI:
    # Momentum indicator between 0 and 100.
    # High RSI usually means strong recent buying; low RSI means weak/oversold behaviour.
    g["rsi_14"] = compute_rsi(g["adj_close"], period=14)

    # Trend flags:
    # Simple binary indicators showing whether price is above key moving averages.
    # 1 = bullish/positive trend condition, 0 = not above that moving average.
    g["above_ma_20"] = (g["adj_close"] > g["ma_20"]).astype(int)
    g["above_ma_50"] = (g["adj_close"] > g["ma_50"]).astype(int)
    g["above_ma_200"] = (g["adj_close"] > g["ma_200"]).astype(int)

    # Moving average alignment flags:
    # These capture trend structure.
    # ma_20_above_ma_50 = short-term trend stronger than medium-term trend.
    # ma_50_above_ma_200 = medium-term trend stronger than long-term trend.
    g["ma_20_above_ma_50"] = (g["ma_20"] > g["ma_50"]).astype(int)
    g["ma_50_above_ma_200"] = (g["ma_50"] > g["ma_200"]).astype(int)

    return g

In [10]:
stock_feature_parts = []

for ticker, g in stocks.groupby("yf_ticker"):
    stock_feature_parts.append(add_base_stock_features(g))

stock_features = pd.concat(stock_feature_parts, ignore_index=True)

stock_features = stock_features.sort_values(["yf_ticker", "date"]).reset_index(drop=True)

print("Stock features shape:", stock_features.shape)

display(
    stock_features[
        [
            "date",
            "symbol",
            "yf_ticker",
            "adj_close",
            "return_1d",
            "return_20d",
            "annualized_volatility_20d",
            "adj_close_to_ma_50",
            "drawdown_from_252d_high",
            "rsi_14",
        ]
    ]
    .dropna()
    .head(10)
)

Stock features shape: (357370, 48)


,date,symbol,yf_ticker,adj_close,return_1d,return_20d,annualized_volatility_20d,adj_close_to_ma_50,drawdown_from_252d_high,rsi_14
251,2011-01-04,ABB,ABB.NS,688.368958,-0.002650,0.007409,0.274947,-0.013184,-0.147111,53.124928
252,2011-01-05,ABB,ABB.NS,677.520630,-0.015759,-0.008591,0.280960,-0.026227,-0.160552,48.771730
253,2011-01-06,ABB,ABB.NS,680.030762,0.003705,0.008708,0.276826,-0.019888,-0.157442,49.796815
254,2011-01-07,ABB,ABB.NS,664.503052,-0.022834,0.027835,0.246445,-0.039444,-0.176681,43.939474
255,2011-01-10,ABB,ABB.NS,637.020996,-0.041357,-0.019063,0.289086,-0.075995,-0.210731,35.892552
256,2011-01-11,ABB,ABB.NS,628.214844,-0.013824,-0.051573,0.281660,-0.085847,-0.221642,33.759079
257,2011-01-12,ABB,ABB.NS,636.510376,0.013205,-0.050273,0.282636,-0.072073,-0.211364,37.526314
258,2011-01-13,ABB,ABB.NS,640.849792,0.006818,-0.030755,0.281299,-0.063670,-0.205988,39.465723
259,2011-01-14,ABB,ABB.NS,638.169495,-0.004182,-0.032693,0.281461,-0.064934,-0.209309,38.667265
260,2011-01-17,ABB,ABB.NS,635.021484,-0.004933,-0.025907,0.279145,-0.066897,-0.213209,37.702460


In [11]:
# to display all the columns in the DataFrame without truncation
pd.set_option("display.max_columns", None)
stock_features.head(10)
print(stock_features.shape)
stock_features.head(10)

(357370, 48)


,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume,source_file,history_days_available,prev_close,prev_adj_close,return_1d,return_5d,return_20d,return_60d,return_120d,log_return_1d,raw_return_1d,daily_volatility_20d,daily_volatility_60d,annualized_volatility_20d,annualized_volatility_60d,ma_20,adj_close_to_ma_20,ma_50,adj_close_to_ma_50,ma_100,adj_close_to_ma_100,ma_200,adj_close_to_ma_200,rolling_high_60d,rolling_high_252d,drawdown_from_60d_high,drawdown_from_252d_high,rsi_14,above_ma_20,above_ma_50,above_ma_200,ma_20_above_ma_50,ma_50_above_ma_200
0,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-04,649.419373,694.875061,0.0,703.408936,690.063416,699.051208,False,0.0,204731,ABB_NS.csv,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
1,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-05,647.765015,693.104736,0.0,700.821533,691.788330,695.419739,False,0.0,175062,ABB_NS.csv,2,694.875061,649.419373,-0.002547,NaN,NaN,NaN,NaN,-0.002551,-0.002548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-06,647.977051,693.331665,0.0,700.367615,690.199585,698.733459,False,0.0,222114,ABB_NS.csv,3,693.104736,647.765015,0.000327,NaN,NaN,NaN,NaN,0.000327,0.000327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
3,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-07,663.758423,710.217834,0.0,717.208374,692.877747,693.604065,False,0.0,438939,ABB_NS.csv,4,693.331665,647.977051,0.024355,NaN,NaN,NaN,NaN,0.024063,0.024355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
4,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-08,675.382690,722.655518,0.0,728.919739,711.171082,713.713135,False,0.0,784364,ABB_NS.csv,5,710.217834,663.758423,0.017513,NaN,NaN,NaN,NaN,0.017361,0.017512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
5,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-11,696.297363,745.034241,0.0,747.167725,724.471252,724.471252,False,0.0,587128,ABB_NS.csv,6,722.655518,675.382690,0.030967,0.072184,NaN,NaN,NaN,0.030497,0.030967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
6,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-12,678.140198,725.606079,0.0,747.167725,721.293762,745.896729,False,0.0,386194,ABB_NS.csv,7,745.034241,696.297363,-0.026077,0.046892,NaN,NaN,NaN,-0.026423,-0.026077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
7,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-13,695.151917,743.808655,0.0,746.986145,716.300537,717.208374,False,0.0,490669,ABB_NS.csv,8,725.606079,678.140198,0.025086,0.072803,NaN,NaN,NaN,0.024776,0.025086,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
8,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-14,724.551270,775.265930,0.0,779.396667,737.816772,747.167725,False,0.0,1181395,ABB_NS.csv,9,743.808655,695.151917,0.042292,0.091589,NaN,NaN,NaN,0.041422,0.042292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
9,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-15,726.545471,777.399414,0.0,794.285583,771.725281,776.219177,False,0.0,842989,ABB_NS.csv,10,775.265930,724.551270,0.002752,0.075754,NaN,NaN,NaN,0.002749,0.002752,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0


### ATR, price range, gap, and volume features (Risk Features)

In [12]:
def add_risk_range_volume_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    # True Range:
    # Measures the actual daily price movement range, including overnight gaps.
    # This is stronger than just high-low because it considers previous close.
    high_low = g["high"] - g["low"]
    high_prev_close = (g["high"] - g["prev_close"]).abs()
    low_prev_close = (g["low"] - g["prev_close"]).abs()

    g["true_range"] = pd.concat(
        [high_low, high_prev_close, low_prev_close],
        axis=1
    ).max(axis=1)

    # ATR:
    # Average True Range over 14 trading days.
    # Used as a practical risk measure for expected price movement.
    g["atr_14"] = g["true_range"].rolling(14).mean()

    # ATR percentage:
    # ATR scaled by current close price.
    # Makes risk comparable across stocks with different prices.
    g["atr_14_pct"] = g["atr_14"] / g["close"]

    # Intraday range:
    # Measures how wide the day's high-low movement was relative to close.
    # Higher value means more unstable intraday behaviour.
    g["intraday_range_pct"] = (g["high"] - g["low"]) / g["close"]

    # Open gap:
    # Measures how much the stock opened above/below previous close.
    # Useful for overnight risk and news shock detection.
    g["open_gap_pct"] = g["open"] / g["prev_close"] - 1

    # Close position in daily range:
    # 0 means close near day's low.
    # 1 means close near day's high.
    # Useful for judging buying/selling pressure within the day.
    g["close_position_in_day_range"] = (
        (g["close"] - g["low"]) / (g["high"] - g["low"]).replace(0, np.nan)
    )

    # Volume moving averages:
    # Smooth normal trading activity over 20 and 60 days.
    g["volume_ma_20"] = g["volume"].rolling(20).mean()
    g["volume_ma_60"] = g["volume"].rolling(60).mean()

    # Volume ratios:
    # Current volume compared to normal volume.
    # High value can indicate unusual interest, panic, breakout, or news activity.
    g["volume_ratio_20d"] = g["volume"] / g["volume_ma_20"]
    g["volume_ratio_60d"] = g["volume"] / g["volume_ma_60"]

    # Turnover:
    # Approximate traded value = close price * volume.
    # Helps measure liquidity better than raw volume alone.
    g["turnover"] = g["close"] * g["volume"]

    # Turnover ratio:
    # Current traded value compared to recent average traded value.
    # Useful for detecting unusual liquidity/activity.
    g["turnover_ma_20"] = g["turnover"].rolling(20).mean()
    g["turnover_ratio_20d"] = g["turnover"] / g["turnover_ma_20"]

    # Large raw move flag:
    # Marks rows where raw close moved more than 20% in one day.
    # These rows are not deleted; they are flagged for caution.
    g["large_raw_move_flag"] = (g["raw_return_1d"].abs() > 0.20).astype(int)

    return g

In [13]:
risk_feature_parts = []

for ticker, g in stock_features.groupby("yf_ticker"):
    risk_feature_parts.append(add_risk_range_volume_features(g))

stock_features = pd.concat(risk_feature_parts, ignore_index=True)
stock_features = stock_features.sort_values(["yf_ticker", "date"]).reset_index(drop=True)

print("Stock features shape:", stock_features.shape)

display(
    stock_features[
        [
            "date",
            "symbol",
            "yf_ticker",
            "close",
            "true_range",
            "atr_14",
            "atr_14_pct",
            "intraday_range_pct",
            "open_gap_pct",
            "close_position_in_day_range",
            "volume_ratio_20d",
            "turnover_ratio_20d",
            "large_raw_move_flag",
        ]
    ]
    .dropna()
    .head(20)
)

Stock features shape: (357370, 62)


,date,symbol,yf_ticker,close,true_range,atr_14,atr_14_pct,intraday_range_pct,open_gap_pct,close_position_in_day_range,volume_ratio_20d,turnover_ratio_20d,large_raw_move_flag
19,2010-02-01,ABB,ABB.NS,738.452271,20.880737,29.913945,0.040509,0.028276,0.000000,0.582608,0.527440,0.523023,0
20,2010-02-02,ABB,ABB.NS,732.914307,19.564392,29.463261,0.040200,0.026694,0.008114,0.229698,0.807242,0.793813,0
21,2010-02-03,ABB,ABB.NS,735.138550,15.206604,28.357618,0.038575,0.020685,0.012883,0.149250,0.591874,0.583266,0
22,2010-02-04,ABB,ABB.NS,719.750366,24.149048,27.112558,0.037669,0.033552,-0.002161,0.251879,0.451629,0.435361,0
23,2010-02-05,ABB,ABB.NS,706.132507,18.883423,26.849923,0.038024,0.021471,-0.005172,0.347305,0.351973,0.332409,0
24,2010-02-08,ABB,ABB.NS,717.889282,27.780457,27.556754,0.038386,0.038697,0.009257,0.754903,0.680155,0.651932,0
25,2010-02-09,ABB,ABB.NS,712.305908,19.065002,27.242244,0.038245,0.026765,-0.000948,0.171427,0.496256,0.472360,0
26,2010-02-10,ABB,ABB.NS,717.117554,18.974243,26.969884,0.037609,0.025636,0.006883,0.229628,0.553445,0.530193,0
27,2010-02-11,ABB,ABB.NS,719.296448,15.932922,24.988804,0.034741,0.021898,0.002912,0.126803,0.297619,0.286031,0
28,2010-02-15,ABB,ABB.NS,720.976013,8.760864,23.211997,0.032195,0.012151,0.000883,0.523318,0.382219,0.370833,0


### flag data useble for training and merging identifing columns

In [14]:
# Keep only stock-level acceptance columns needed for feature table
stock_acceptance_small = (
    stock_acceptance[
        [
            "symbol",
            "yf_ticker",
            "status",
            "rows",
            "start_date",
            "end_date",
        ]
    ]
    .drop_duplicates("yf_ticker")
    .rename(
        columns={
            "status": "stock_acceptance_status",
            "rows": "stock_history_rows",
            "start_date": "stock_history_start_date",
            "end_date": "stock_history_end_date",
        }
    )
)

# Merge stock-level quality status into every daily stock row
stock_features = stock_features.merge(
    stock_acceptance_small,
    on=["symbol", "yf_ticker"],
    how="left"
)

# Research universe:
# 1 means stock has enough history and passed quality checks.
# This will be used for training/backtesting.
stock_features["is_research_universe"] = (
    stock_features["stock_acceptance_status"].eq("OK").astype(int)
)

# Limited history:
# 1 means stock is usable for dashboard/current analysis,
# but weaker for long historical backtesting.
stock_features["is_limited_history"] = (
    stock_features["stock_acceptance_status"].eq("FLAG_SHORT_HISTORY").astype(int)
)

print("Stock features shape:", stock_features.shape)

display(stock_features["stock_acceptance_status"].value_counts(dropna=False))

display(
    stock_features[
        [
            "symbol",
            "yf_ticker",
            "date",
            "stock_acceptance_status",
            "stock_history_rows",
            "stock_history_start_date",
            "stock_history_end_date",
            "is_research_universe",
            "is_limited_history",
        ]
    ]
    .drop_duplicates(["symbol", "yf_ticker"])
    .sort_values(["stock_acceptance_status", "symbol"])
    .head(120)
)

Stock features shape: (357370, 68)


stock_acceptance_status
OK                    348833
FLAG_SHORT_HISTORY      8537
Name: count, dtype: int64

,symbol,yf_ticker,date,stock_acceptance_status,stock_history_rows,stock_history_start_date,stock_history_end_date,is_research_universe,is_limited_history
121087,ENRIN,ENRIN.NS,2025-06-19,FLAG_SHORT_HISTORY,267,2025-06-19,2026-07-14,0,1
121354,ETERNAL,ETERNAL.NS,2021-07-23,FLAG_SHORT_HISTORY,1232,2021-07-23,2026-07-14,0,1
161391,HYUNDAI,HYUNDAI.NS,2024-10-22,FLAG_SHORT_HISTORY,430,2024-10-22,2026-07-14,0,1
180785,IRFC,IRFC.NS,2021-01-29,FLAG_SHORT_HISTORY,1350,2021-01-29,2026-07-14,0,1
190299,JIOFIN,JIOFIN.NS,2023-08-21,FLAG_SHORT_HISTORY,717,2023-08-21,2026-07-14,0,1
...,...,...,...,...,...,...,...,...,...
338647,UNITDSPR,UNITDSPR.NS,2010-01-04,OK,4082,2010-01-04,2026-07-14,1,0
342729,VBL,VBL.NS,2016-11-08,OK,2395,2016-11-08,2026-07-14,1,0
345124,VEDL,VEDL.NS,2010-01-04,OK,4082,2010-01-04,2026-07-14,1,0
349206,WIPRO,WIPRO.NS,2010-01-04,OK,4082,2010-01-04,2026-07-14,1,0


### Merge large-move review flags

In [15]:
# Keep only row-level large move review columns needed for feature table
large_move_review_small = (
    large_move_review[
        [
            "date",
            "yf_ticker",
            "large_move_reason",
            "review_status",
        ]
    ]
    .drop_duplicates(["date", "yf_ticker"])
)

# Merge large-move review info into daily stock feature rows
stock_features = stock_features.merge(
    large_move_review_small,
    on=["date", "yf_ticker"],
    how="left"
)

# large_move_review_flag:
# 1 means this row was identified in our >20% daily move review report.
stock_features["large_move_review_flag"] = (
    stock_features["review_status"].notna().astype(int)
)

# manual_review_large_move_flag:
# 1 means the row had a very large move and still needs caution/manual review.
# We keep it for now, but this flag allows us to exclude/test later.
stock_features["manual_review_large_move_flag"] = (
    stock_features["review_status"].eq("MANUAL_REVIEW").astype(int)
)

# keep_large_move_flag:
# 1 means the large move was likely a real market/sector event.
stock_features["keep_large_move_flag"] = (
    stock_features["review_status"].eq("KEEP").astype(int)
)

print("Stock features shape:", stock_features.shape)

display(stock_features["review_status"].value_counts(dropna=False))

display(
    stock_features[
        stock_features["large_move_review_flag"] == 1
    ][
        [
            "date",
            "symbol",
            "yf_ticker",
            "raw_return_1d",
            "large_move_reason",
            "review_status",
            "large_move_review_flag",
            "manual_review_large_move_flag",
            "keep_large_move_flag",
        ]
    ]
    .sort_values("raw_return_1d", key=lambda x: x.abs(), ascending=False)
    .head(30)
)

Stock features shape: (357370, 73)


review_status
NaN              357319
MANUAL_REVIEW        40
KEEP                 11
Name: count, dtype: int64

,date,symbol,yf_ticker,raw_return_1d,large_move_reason,review_status,large_move_review_flag,manual_review_large_move_flag,keep_large_move_flag
225912,2010-01-08,NESTLEIND,NESTLEIND.NS,-0.763338,REAL_OR_NEEDS_REVIEW,MANUAL_REVIEW,1,1,0
349152,2026-04-30,VEDL,VEDL.NS,-0.648979,REAL_OR_NEEDS_REVIEW,MANUAL_REVIEW,1,1,0
248244,2017-10-25,PNB,PNB.NS,0.462012,REAL_MARKET_EVENT_LIKELY,KEEP,1,0,1
318050,2025-10-14,TMPV,TMPV.NS,-0.401513,REAL_OR_NEEDS_REVIEW,MANUAL_REVIEW,1,1,0
8122,2015-06-03,ADANIENT,ADANIENT.NS,-0.387493,REAL_OR_NEEDS_REVIEW,MANUAL_REVIEW,1,1,0
80101,2017-10-25,CANBK,CANBK.NS,0.387260,REAL_MARKET_EVENT_LIKELY,KEEP,1,0,1
339356,2012-11-12,UNITDSPR,UNITDSPR.NS,0.347262,REAL_OR_NEEDS_REVIEW,MANUAL_REVIEW,1,1,0
336491,2017-10-25,UNIONBANK,UNIONBANK.NS,0.341705,REAL_MARKET_EVENT_LIKELY,KEEP,1,0,1
55610,2017-10-25,BANKBARODA,BANKBARODA.NS,0.314356,REAL_MARKET_EVENT_LIKELY,KEEP,1,0,1
88857,2020-03-23,CHOLAFIN,CHOLAFIN.NS,-0.295990,REAL_MARKET_EVENT_LIKELY,KEEP,1,0,1


### index prefix mapping and verify all 16 indices

In [16]:
INDEX_PREFIX_MAP = {
    "^NSEI": "nifty50",
    "^CNX100": "nifty100",
    "^INDIAVIX": "indiavix",
    "^NSEBANK": "nifty_bank",
    "^CNXAUTO": "nifty_auto",
    "^CNXIT": "nifty_it",
    "^CNXENERGY": "nifty_energy",
    "^CNXFMCG": "nifty_fmcg",
    "^CNXMETAL": "nifty_metal",
    "^CNXPHARMA": "nifty_pharma",
    "^CNXPSUBANK": "nifty_psu_bank",
    "^CNXREALTY": "nifty_realty",
    "^CNXCMDT": "nifty_commodities",
    "^CNXMEDIA": "nifty_media",
    "^CNXMNC": "nifty_mnc",
    "^CNXPSE": "nifty_pse",
}

available_index_tickers = sorted(indices["yf_ticker"].unique())
mapped_index_tickers = sorted(INDEX_PREFIX_MAP.keys())

unmapped_indices = sorted(set(available_index_tickers) - set(mapped_index_tickers))
missing_from_data = sorted(set(mapped_index_tickers) - set(available_index_tickers))

print("Available index tickers:", len(available_index_tickers))
print("Mapped index tickers:", len(mapped_index_tickers))
print("Unmapped indices in data:", unmapped_indices)
print("Mapped indices missing from data:", missing_from_data)

display(
    pd.DataFrame(
        {
            "yf_ticker": available_index_tickers,
            "prefix": [INDEX_PREFIX_MAP.get(ticker) for ticker in available_index_tickers],
        }
    )
)

Available index tickers: 16
Mapped index tickers: 16
Unmapped indices in data: []
Mapped indices missing from data: []


,yf_ticker,prefix
0,^CNX100,nifty100
1,^CNXAUTO,nifty_auto
2,^CNXCMDT,nifty_commodities
3,^CNXENERGY,nifty_energy
4,^CNXFMCG,nifty_fmcg
5,^CNXIT,nifty_it
6,^CNXMEDIA,nifty_media
7,^CNXMETAL,nifty_metal
8,^CNXMNC,nifty_mnc
9,^CNXPHARMA,nifty_pharma


### index feature function

In [17]:
def add_index_features(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("date").copy()

    # idx_close:
    # Standard close value for the index.
    # For indices, we use close directly because adjusted close is usually not needed.
    g["idx_close"] = g["close"]

    # Index return features:
    # These measure market/sector movement over different time windows.
    # Used to understand whether the overall market or sector is rising/falling.
    g["idx_return_1d"] = g["idx_close"].pct_change(1)
    g["idx_return_5d"] = g["idx_close"].pct_change(5)
    g["idx_return_20d"] = g["idx_close"].pct_change(20)
    g["idx_return_60d"] = g["idx_close"].pct_change(60)

    # Index volatility:
    # Measures how unstable the market/sector has been recently.
    # Useful because stock risk is usually higher when market/sector volatility is high.
    g["idx_daily_volatility_20d"] = g["idx_return_1d"].rolling(20).std()
    g["idx_daily_volatility_60d"] = g["idx_return_1d"].rolling(60).std()

    # Annualized index volatility:
    # Converts daily volatility to yearly scale using 252 trading days.
    g["idx_annualized_volatility_20d"] = g["idx_daily_volatility_20d"] * np.sqrt(252)
    g["idx_annualized_volatility_60d"] = g["idx_daily_volatility_60d"] * np.sqrt(252)

    # Index moving average distance:
    # Positive value = index is above average.
    # Negative value = index is below average.
    for window in [20, 50, 200]:
        ma_col = f"idx_ma_{window}"
        ratio_col = f"idx_close_to_ma_{window}"

        g[ma_col] = g["idx_close"].rolling(window).mean()
        g[ratio_col] = g["idx_close"] / g[ma_col] - 1

    return g    

### testing on nifty50

In [18]:
nifty_sample = indices[indices["yf_ticker"] == "^NSEI"].copy()

nifty_sample_features = add_index_features(nifty_sample)

display(
    nifty_sample_features[
        [
            "date",
            "yf_ticker",
            "idx_close",
            "idx_return_1d",
            "idx_return_20d",
            "idx_annualized_volatility_20d",
            "idx_close_to_ma_50",
            "idx_close_to_ma_200",
        ]
    ]
    .dropna()
    .head(20)
)

,date,yf_ticker,idx_close,idx_return_1d,idx_return_20d,idx_annualized_volatility_20d,idx_close_to_ma_50,idx_close_to_ma_200
57832,2010-10-19,^NSEI,6027.299805,-0.008007,0.003037,0.158096,0.042599,0.132355
57833,2010-10-20,^NSEI,5982.100098,-0.007499,-0.001486,0.160096,0.032917,0.123073
57834,2010-10-21,^NSEI,6101.500000,0.019960,0.023819,0.173624,0.051063,0.144604
57835,2010-10-22,^NSEI,6066.049805,-0.005810,0.007934,0.172204,0.042622,0.137117
57836,2010-10-25,^NSEI,6105.799805,0.006553,0.011623,0.173380,0.047102,0.143665
57837,2010-10-26,^NSEI,6082.000000,-0.003898,0.008707,0.174044,0.040651,0.138315
57838,2010-10-27,^NSEI,6012.649902,-0.011403,0.003564,0.177573,0.026682,0.124532
57839,2010-10-28,^NSEI,5987.700195,-0.004150,-0.007007,0.176637,0.020649,0.119052
57840,2010-10-29,^NSEI,6017.700195,0.005010,-0.020461,0.163111,0.024096,0.123836
57841,2010-11-01,^NSEI,6117.549805,0.016593,-0.006803,0.174359,0.039013,0.141569


### Create wide index feature table

In [19]:
index_feature_frames = []

index_feature_cols = [
    "idx_close",
    "idx_return_1d",
    "idx_return_5d",
    "idx_return_20d",
    "idx_return_60d",
    "idx_daily_volatility_20d",
    "idx_daily_volatility_60d",
    "idx_annualized_volatility_20d",
    "idx_annualized_volatility_60d",
    "idx_close_to_ma_20",
    "idx_close_to_ma_50",
    "idx_close_to_ma_200",
]

for ticker, g in indices.groupby("yf_ticker"):
    prefix = INDEX_PREFIX_MAP.get(ticker)

    if prefix is None:
        print("Skipping unmapped index:", ticker)
        continue

    gf = add_index_features(g)

    keep_cols = ["date"] + [col for col in index_feature_cols if col in gf.columns]
    gf = gf[keep_cols].copy()

    rename_map = {
        col: f"{prefix}_{col.replace('idx_', '')}"
        for col in gf.columns
        if col != "date"
    }

    gf = gf.rename(columns=rename_map)

    index_feature_frames.append(gf)

index_features_wide = None

for gf in index_feature_frames:
    if index_features_wide is None:
        index_features_wide = gf
    else:
        index_features_wide = index_features_wide.merge(gf, on="date", how="outer")

index_features_wide = index_features_wide.sort_values("date").reset_index(drop=True)

print("Index features wide shape:", index_features_wide.shape)
print("Date range:", index_features_wide["date"].min(), "to", index_features_wide["date"].max())

display(index_features_wide.head())
display(index_features_wide.tail())

Index features wide shape: (4074, 193)
Date range: 2010-01-04 00:00:00 to 2026-07-14 00:00:00


,date,nifty100_close,nifty100_return_1d,nifty100_return_5d,nifty100_return_20d,nifty100_return_60d,nifty100_daily_volatility_20d,nifty100_daily_volatility_60d,nifty100_annualized_volatility_20d,nifty100_annualized_volatility_60d,nifty100_close_to_ma_20,nifty100_close_to_ma_50,nifty100_close_to_ma_200,nifty_auto_close,nifty_auto_return_1d,nifty_auto_return_5d,nifty_auto_return_20d,nifty_auto_return_60d,nifty_auto_daily_volatility_20d,nifty_auto_daily_volatility_60d,nifty_auto_annualized_volatility_20d,nifty_auto_annualized_volatility_60d,nifty_auto_close_to_ma_20,nifty_auto_close_to_ma_50,nifty_auto_close_to_ma_200,nifty_commodities_close,nifty_commodities_return_1d,nifty_commodities_return_5d,nifty_commodities_return_20d,nifty_commodities_return_60d,nifty_commodities_daily_volatility_20d,nifty_commodities_daily_volatility_60d,nifty_commodities_annualized_volatility_20d,nifty_commodities_annualized_volatility_60d,nifty_commodities_close_to_ma_20,nifty_commodities_close_to_ma_50,nifty_commodities_close_to_ma_200,nifty_energy_close,nifty_energy_return_1d,nifty_energy_return_5d,nifty_energy_return_20d,nifty_energy_return_60d,nifty_energy_daily_volatility_20d,nifty_energy_daily_volatility_60d,nifty_energy_annualized_volatility_20d,nifty_energy_annualized_volatility_60d,nifty_energy_close_to_ma_20,nifty_energy_close_to_ma_50,nifty_energy_close_to_ma_200,nifty_fmcg_close,nifty_fmcg_return_1d,nifty_fmcg_return_5d,nifty_fmcg_return_20d,nifty_fmcg_return_60d,nifty_fmcg_daily_volatility_20d,nifty_fmcg_daily_volatility_60d,nifty_fmcg_annualized_volatility_20d,nifty_fmcg_annualized_volatility_60d,nifty_fmcg_close_to_ma_20,nifty_fmcg_close_to_ma_50,nifty_fmcg_close_to_ma_200,nifty_it_close,nifty_it_return_1d,nifty_it_return_5d,nifty_it_return_20d,nifty_it_return_60d,nifty_it_daily_volatility_20d,nifty_it_daily_volatility_60d,nifty_it_annualized_volatility_20d,nifty_it_annualized_volatility_60d,nifty_it_close_to_ma_20,nifty_it_close_to_ma_50,nifty_it_close_to_ma_200,nifty_media_close,nifty_media_return_1d,nifty_media_return_5d,nifty_media_return_20d,nifty_media_return_60d,nifty_media_daily_volatility_20d,nifty_media_daily_volatility_60d,nifty_media_annualized_volatility_20d,nifty_media_annualized_volatility_60d,nifty_media_close_to_ma_20,nifty_media_close_to_ma_50,nifty_media_close_to_ma_200,nifty_metal_close,nifty_metal_return_1d,nifty_metal_return_5d,nifty_metal_return_20d,nifty_metal_return_60d,nifty_metal_daily_volatility_20d,nifty_metal_daily_volatility_60d,nifty_metal_annualized_volatility_20d,nifty_metal_annualized_volatility_60d,nifty_metal_close_to_ma_20,nifty_metal_close_to_ma_50,nifty_metal_close_to_ma_200,nifty_mnc_close,nifty_mnc_return_1d,nifty_mnc_return_5d,nifty_mnc_return_20d,nifty_mnc_return_60d,nifty_mnc_daily_volatility_20d,nifty_mnc_daily_volatility_60d,nifty_mnc_annualized_volatility_20d,nifty_mnc_annualized_volatility_60d,nifty_mnc_close_to_ma_20,nifty_mnc_close_to_ma_50,nifty_mnc_close_to_ma_200,nifty_pharma_close,nifty_pharma_return_1d,nifty_pharma_return_5d,nifty_pharma_return_20d,nifty_pharma_return_60d,nifty_pharma_daily_volatility_20d,nifty_pharma_daily_volatility_60d,nifty_pharma_annualized_volatility_20d,nifty_pharma_annualized_volatility_60d,nifty_pharma_close_to_ma_20,nifty_pharma_close_to_ma_50,nifty_pharma_close_to_ma_200,nifty_pse_close,nifty_pse_return_1d,nifty_pse_return_5d,nifty_pse_return_20d,nifty_pse_return_60d,nifty_pse_daily_volatility_20d,nifty_pse_daily_volatility_60d,nifty_pse_annualized_volatility_20d,nifty_pse_annualized_volatility_60d,nifty_pse_close_to_ma_20,nifty_pse_close_to_ma_50,nifty_pse_close_to_ma_200,nifty_psu_bank_close,nifty_psu_bank_return_1d,nifty_psu_bank_return_5d,nifty_psu_bank_return_20d,nifty_psu_bank_return_60d,nifty_psu_bank_daily_volatility_20d,nifty_psu_bank_daily_volatility_60d,nifty_psu_bank_annualized_volatility_20d,nifty_psu_bank_annualized_volatility_60d,nifty_psu_bank_close_to_ma_20,nifty_psu_bank_close_to_ma_50,nifty_psu_bank_close_to_ma_200,nifty_realty_close,nif

,date,nifty100_close,nifty100_return_1d,nifty100_return_5d,nifty100_return_20d,nifty100_return_60d,nifty100_daily_volatility_20d,nifty100_daily_volatility_60d,nifty100_annualized_volatility_20d,nifty100_annualized_volatility_60d,nifty100_close_to_ma_20,nifty100_close_to_ma_50,nifty100_close_to_ma_200,nifty_auto_close,nifty_auto_return_1d,nifty_auto_return_5d,nifty_auto_return_20d,nifty_auto_return_60d,nifty_auto_daily_volatility_20d,nifty_auto_daily_volatility_60d,nifty_auto_annualized_volatility_20d,nifty_auto_annualized_volatility_60d,nifty_auto_close_to_ma_20,nifty_auto_close_to_ma_50,nifty_auto_close_to_ma_200,nifty_commodities_close,nifty_commodities_return_1d,nifty_commodities_return_5d,nifty_commodities_return_20d,nifty_commodities_return_60d,nifty_commodities_daily_volatility_20d,nifty_commodities_daily_volatility_60d,nifty_commodities_annualized_volatility_20d,nifty_commodities_annualized_volatility_60d,nifty_commodities_close_to_ma_20,nifty_commodities_close_to_ma_50,nifty_commodities_close_to_ma_200,nifty_energy_close,nifty_energy_return_1d,nifty_energy_return_5d,nifty_energy_return_20d,nifty_energy_return_60d,nifty_energy_daily_volatility_20d,nifty_energy_daily_volatility_60d,nifty_energy_annualized_volatility_20d,nifty_energy_annualized_volatility_60d,nifty_energy_close_to_ma_20,nifty_energy_close_to_ma_50,nifty_energy_close_to_ma_200,nifty_fmcg_close,nifty_fmcg_return_1d,nifty_fmcg_return_5d,nifty_fmcg_return_20d,nifty_fmcg_return_60d,nifty_fmcg_daily_volatility_20d,nifty_fmcg_daily_volatility_60d,nifty_fmcg_annualized_volatility_20d,nifty_fmcg_annualized_volatility_60d,nifty_fmcg_close_to_ma_20,nifty_fmcg_close_to_ma_50,nifty_fmcg_close_to_ma_200,nifty_it_close,nifty_it_return_1d,nifty_it_return_5d,nifty_it_return_20d,nifty_it_return_60d,nifty_it_daily_volatility_20d,nifty_it_daily_volatility_60d,nifty_it_annualized_volatility_20d,nifty_it_annualized_volatility_60d,nifty_it_close_to_ma_20,nifty_it_close_to_ma_50,nifty_it_close_to_ma_200,nifty_media_close,nifty_media_return_1d,nifty_media_return_5d,nifty_media_return_20d,nifty_media_return_60d,nifty_media_daily_volatility_20d,nifty_media_daily_volatility_60d,nifty_media_annualized_volatility_20d,nifty_media_annualized_volatility_60d,nifty_media_close_to_ma_20,nifty_media_close_to_ma_50,nifty_media_close_to_ma_200,nifty_metal_close,nifty_metal_return_1d,nifty_metal_return_5d,nifty_metal_return_20d,nifty_metal_return_60d,nifty_metal_daily_volatility_20d,nifty_metal_daily_volatility_60d,nifty_metal_annualized_volatility_20d,nifty_metal_annualized_volatility_60d,nifty_metal_close_to_ma_20,nifty_metal_close_to_ma_50,nifty_metal_close_to_ma_200,nifty_mnc_close,nifty_mnc_return_1d,nifty_mnc_return_5d,nifty_mnc_return_20d,nifty_mnc_return_60d,nifty_mnc_daily_volatility_20d,nifty_mnc_daily_volatility_60d,nifty_mnc_annualized_volatility_20d,nifty_mnc_annualized_volatility_60d,nifty_mnc_close_to_ma_20,nifty_mnc_close_to_ma_50,nifty_mnc_close_to_ma_200,nifty_pharma_close,nifty_pharma_return_1d,nifty_pharma_return_5d,nifty_pharma_return_20d,nifty_pharma_return_60d,nifty_pharma_daily_volatility_20d,nifty_pharma_daily_volatility_60d,nifty_pharma_annualized_volatility_20d,nifty_pharma_annualized_volatility_60d,nifty_pharma_close_to_ma_20,nifty_pharma_close_to_ma_50,nifty_pharma_close_to_ma_200,nifty_pse_close,nifty_pse_return_1d,nifty_pse_return_5d,nifty_pse_return_20d,nifty_pse_return_60d,nifty_pse_daily_volatility_20d,nifty_pse_daily_volatility_60d,nifty_pse_annualized_volatility_20d,nifty_pse_annualized_volatility_60d,nifty_pse_close_to_ma_20,nifty_pse_close_to_ma_50,nifty_pse_close_to_ma_200,nifty_psu_bank_close,nifty_psu_bank_return_1d,nifty_psu_bank_return_5d,nifty_psu_bank_return_20d,nifty_psu_bank_return_60d,nifty_psu_bank_daily_volatility_20d,nifty_psu_bank_daily_volatility_60d,nifty_psu_bank_annualized_volatility_20d,nifty_psu_bank_annualized_volatility_60d,nifty_psu_bank_close_to_ma_20,nifty_psu_bank_close_to_ma_50,nifty_psu_bank_close_to_ma_200,nifty_realty_close,nif

### Merge only global market features 

In [20]:
global_prefixes = [
    "nifty50",
    "nifty100",
    "indiavix",
]

# Select only global market columns from index_features_wide.
# We are NOT merging all sector index columns into every stock.
global_market_cols = ["date"]

for prefix in global_prefixes:
    matching_cols = [
        col for col in index_features_wide.columns
        if col.startswith(prefix + "_")
    ]
    global_market_cols.extend(matching_cols)

global_market_features = index_features_wide[global_market_cols].copy()

print("Global market features shape:", global_market_features.shape)
print("Global market columns:", len(global_market_features.columns))

display(global_market_features.head())
display(global_market_features.tail())

Global market features shape: (4074, 37)
Global market columns: 37


,date,nifty50_close,nifty50_return_1d,nifty50_return_5d,nifty50_return_20d,nifty50_return_60d,nifty50_daily_volatility_20d,nifty50_daily_volatility_60d,nifty50_annualized_volatility_20d,nifty50_annualized_volatility_60d,nifty50_close_to_ma_20,nifty50_close_to_ma_50,nifty50_close_to_ma_200,nifty100_close,nifty100_return_1d,nifty100_return_5d,nifty100_return_20d,nifty100_return_60d,nifty100_daily_volatility_20d,nifty100_daily_volatility_60d,nifty100_annualized_volatility_20d,nifty100_annualized_volatility_60d,nifty100_close_to_ma_20,nifty100_close_to_ma_50,nifty100_close_to_ma_200,indiavix_close,indiavix_return_1d,indiavix_return_5d,indiavix_return_20d,indiavix_return_60d,indiavix_daily_volatility_20d,indiavix_daily_volatility_60d,indiavix_annualized_volatility_20d,indiavix_annualized_volatility_60d,indiavix_close_to_ma_20,indiavix_close_to_ma_50,indiavix_close_to_ma_200
0,2010-01-04,5232.200195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5153.549805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.639999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,5277.899902,0.008734,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5203.799805,0.009751,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.270000,-0.057953,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-06,5281.799805,0.000739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5214.000000,0.001960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.120001,-0.006736,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-07,5263.100098,-0.003540,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5191.700195,-0.004277,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.500000,0.017179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-01-08,5244.750000,-0.003487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5175.750000,-0.003072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.570000,0.003111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,date,nifty50_close,nifty50_return_1d,nifty50_return_5d,nifty50_return_20d,nifty50_return_60d,nifty50_daily_volatility_20d,nifty50_daily_volatility_60d,nifty50_annualized_volatility_20d,nifty50_annualized_volatility_60d,nifty50_close_to_ma_20,nifty50_close_to_ma_50,nifty50_close_to_ma_200,nifty100_close,nifty100_return_1d,nifty100_return_5d,nifty100_return_20d,nifty100_return_60d,nifty100_daily_volatility_20d,nifty100_daily_volatility_60d,nifty100_annualized_volatility_20d,nifty100_annualized_volatility_60d,nifty100_close_to_ma_20,nifty100_close_to_ma_50,nifty100_close_to_ma_200,indiavix_close,indiavix_return_1d,indiavix_return_5d,indiavix_return_20d,indiavix_return_60d,indiavix_daily_volatility_20d,indiavix_daily_volatility_60d,indiavix_annualized_volatility_20d,indiavix_annualized_volatility_60d,indiavix_close_to_ma_20,indiavix_close_to_ma_50,indiavix_close_to_ma_200
4069,2026-07-08,23882.050781,-0.021175,-0.005157,0.027534,0.004498,0.008600,0.008324,0.136516,0.132141,-0.003023,0.002267,-0.039265,24883.050781,-0.020724,-0.007263,0.025023,0.016195,0.008821,0.008574,0.140030,0.136111,-0.004402,0.002295,-0.027188,14.68,0.260086,0.108761,-0.057766,-0.281449,0.071707,0.060418,1.138309,0.959101,0.093849,-0.071709,0.026599
4070,2026-07-09,23962.800781,0.003381,-0.008806,0.032214,-0.003651,0.008588,0.008199,0.136338,0.130152,-0.001211,0.005766,-0.035794,24989.250000,0.004268,-0.009550,0.032582,0.007233,0.008783,0.008423,0.139426,0.133704,-0.001728,0.006623,-0.022892,13.36,-0.089918,0.087063,-0.145234,-0.291247,0.074408,0.060698,1.181183,0.963558,0.003983,-0.149781,-0.066769
4071,2026-07-10,24206.900391,0.010187,-0.002635,0.045131,0.015277,0.008741,0.008224,0.138758,0.130547,0.006770,0.015831,-0.025803,25255.099609,0.010639,-0.001850,0.047725,0.026645,0.008899,0.008452,0.141260,0.134174,0.006579,0.017087,-0.012404,12.25,-0.083084,0.038136,-0.215247,-0.402439,0.076390,0.060333,1.212647,0.957762,-0.067661,-0.214623,-0.144858
4072,2026-07-13,24211.000000,0.000169,-0.008979,0.024895,-0.000838,0.007692,0.007950,0.122111,0.126209,0.005710,0.015975,-0.025436,25240.800781,-0.000566,-0.008783,0.025955,0.008676,0.007803,0.008161,0.123869,0.129554,0.004731,0.016400,-0.012835,13.28,0.084082,0.123519,-0.097826,-0.288698,0.078264,0.060470,1.242410,0.959924,0.016301,-0.144021,-0.073929
4073,2026-07-14,24052.050781,-0.006565,-0.014208,0.008307,-0.005980,0.007607,0.007994,0.120760,0.126897,-0.001303,0.009258,-0.031585,25088.650391,-0.006028,-0.012633,0.007259,0.001851,0.007505,0.008200,0.119143,0.130169,-0.001685,0.010117,-0.018613,13.75,0.035392,0.180258,-0.041812,-0.239912,0.078511,0.060573,1.246330,0.961566,0.054691,-0.108312,-0.042322


In [21]:
print("Stock features shape before global market merge:", stock_features.shape)
stock_features = stock_features.merge(
    global_market_features,
    on="date",
    how="left"
)

print("Stock features shape after global market merge:", stock_features.shape)

important_global_cols = [
    "nifty50_close",
    "nifty50_return_20d",
    "nifty100_close",
    "nifty100_return_20d",
    "indiavix_close",
    "indiavix_return_20d",
]

missing_global_cols = [
    col for col in important_global_cols
    if col not in stock_features.columns
]

print("Missing global columns:", missing_global_cols)

display(
    stock_features[
        [
            "date",
            "symbol",
            "yf_ticker",
            "return_20d",
            "nifty50_return_20d",
            "nifty100_return_20d",
            "indiavix_close",
        ]
    ]
    .dropna()
    .head(20)
)

Stock features shape before global market merge: (357370, 73)
Stock features shape after global market merge: (357370, 109)
Missing global columns: []


,date,symbol,yf_ticker,return_20d,nifty50_return_20d,nifty100_return_20d,indiavix_close
20,2010-02-02,ABB,ABB.NS,0.054743,-0.076851,-0.071737,26.690001
21,2010-02-03,ABB,ABB.NS,0.060646,-0.065566,-0.061340,25.540001
22,2010-02-04,ABB,ABB.NS,0.038104,-0.082633,-0.079814,27.070000
23,2010-02-05,ABB,ABB.NS,-0.005752,-0.103447,-0.099553,30.070000
24,2010-02-08,ABB,ABB.NS,-0.006595,-0.092350,-0.087929,30.360001
25,2010-02-09,ABB,ABB.NS,-0.043929,-0.087010,-0.083995,29.910000
26,2010-02-10,ABB,ABB.NS,-0.011699,-0.086980,-0.082758,30.549999
27,2010-02-11,ABB,ABB.NS,-0.032955,-0.077781,-0.074762,28.809999
28,2010-02-15,ABB,ABB.NS,-0.070027,-0.087064,-0.084425,29.680000
29,2010-02-16,ABB,ABB.NS,-0.064931,-0.075483,-0.073166,28.730000


In [22]:
stock_features.sample(5)

,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume,source_file,history_days_available,prev_close,prev_adj_close,return_1d,return_5d,return_20d,return_60d,return_120d,log_return_1d,raw_return_1d,daily_volatility_20d,daily_volatility_60d,annualized_volatility_20d,annualized_volatility_60d,ma_20,adj_close_to_ma_20,ma_50,adj_close_to_ma_50,ma_100,adj_close_to_ma_100,ma_200,adj_close_to_ma_200,rolling_high_60d,rolling_high_252d,drawdown_from_60d_high,drawdown_from_252d_high,rsi_14,above_ma_20,above_ma_50,above_ma_200,ma_20_above_ma_50,ma_50_above_ma_200,true_range,atr_14,atr_14_pct,intraday_range_pct,open_gap_pct,close_position_in_day_range,volume_ma_20,volume_ma_60,volume_ratio_20d,volume_ratio_60d,turnover,turnover_ma_20,turnover_ratio_20d,large_raw_move_flag,stock_acceptance_status,stock_history_rows,stock_history_start_date,stock_history_end_date,is_research_universe,is_limited_history,large_move_reason,review_status,large_move_review_flag,manual_review_large_move_flag,keep_large_move_flag,nifty50_close,nifty50_return_1d,nifty50_return_5d,nifty50_return_20d,nifty50_return_60d,nifty50_daily_volatility_20d,nifty50_daily_volatility_60d,nifty50_annualized_volatility_20d,nifty50_annualized_volatility_60d,nifty50_close_to_ma_20,nifty50_close_to_ma_50,nifty50_close_to_ma_200,nifty100_close,nifty100_return_1d,nifty100_return_5d,nifty100_return_20d,nifty100_return_60d,nifty100_daily_volatility_20d,nifty100_daily_volatility_60d,nifty100_annualized_volatility_20d,nifty100_annualized_volatility_60d,nifty100_close_to_ma_20,nifty100_close_to_ma_50,nifty100_close_to_ma_200,indiavix_close,indiavix_return_1d,indiavix_return_5d,indiavix_return_20d,indiavix_return_60d,indiavix_daily_volatility_20d,indiavix_daily_volatility_60d,indiavix_annualized_volatility_20d,indiavix_annualized_volatility_60d,indiavix_close_to_ma_20,indiavix_close_to_ma_50,indiavix_close_to_ma_200
49684,stock,BAJFINANCE,BAJFINANCE.NS,Bajaj Finance Ltd.,Financial Services,2010-05-06,3.789855,4.197716,0.0,4.345397,4.161768,4.304104,False,0.0,6508359,BAJFINANCE_NS.csv,83,4.297789,3.880206,-0.023285,0.015155,0.286629,0.444258,NaN,-0.023560,-0.023285,0.023129,0.021863,0.367164,0.347063,3.459115,0.095614,3.037060,0.247870,NaN,NaN,NaN,NaN,3.880206,NaN,-0.023285,NaN,76.094114,1,1,0,1,0,0.183629,0.244526,0.058252,0.043745,0.001469,0.195766,7539487.30,4.011299e+06,0.863236,1.622507,2.732024e+07,2.991128e+07,0.913376,0,OK,4082,2010-01-04,2026-07-14,1,0,NaN,NaN,0,0,0,5090.850098,-0.006644,-0.031080,-0.052803,0.050667,0.008154,0.008725,0.129444,0.138512,-0.031076,-0.019445,NaN,5067.649902,-0.006460,-0.026098,-0.046404,0.056233,0.007855,0.008491,0.124689,0.134789,-0.026455,-0.013025,NaN,24.360001,0.028282,0.149599,0.406467,-0.100111,0.069553,0.063050,1.104118,1.000883,0.152372,0.160553,NaN
118871,stock,EICHERMOT,EICHERMOT.NS,Eicher Motors Ltd.,Automobile and Auto Components,2017-07-27,2712.475098,2910.225098,0.0,3016.250000,2856.985107,2899.000000,False,0.0,1117140,EICHERMOT_NS.csv,1867,2900.245117,2703.173340,0.003441,-0.005141,0.064455,0.122991,0.211098,0.003435,0.003441,0.012088,0.015766,0.191893,0.250275,2637.394421,0.028468,2640.205483,0.027373,2515.003972,0.078517,2334.985841,0.161667,2774.232178,2774.232178,-0.022261,-0.022261,59.429331,1,1,1,0,1,159.264893,59.644618,0.020495,0.054726,-0.000429,0.334286,395181.50,4.391958e+05,2.826904,2.543603,3.251129e+09,1.120934e+09,2.900376,0,OK,4082,2010-01-04,2026-07-14,1,0,NaN,NaN,0,0,0,10020.549805,-0.000010,0.014914,0.054340,0.076096,0.004578,0.004952,0.072678,0.078606,0.021037,0.036241,0.122029,10348.700195,-0.000864,0.012851,0.055016,0.070585,0.004611,0.005171,0.073205,0.082094,0.019925,0.036335,0.123397,11.220000,0.004476,-0.008834,-0.014060,-0.026886,0.020346,0.037682,0.322977,0.598186,0.003084,-0.003198,-0.169532
140347,stock,HCLTECH,HCLTECH.NS,HCL Technologies Ltd.,Information Technology,2024-01-16,1409.273438,1555.449951,0.0,1587.000000,1538.000000,1587.000000,

### lets see unique indusrties in all stocks 

In [23]:
industry_summary = (
    stocks[["symbol", "yf_ticker", "company_name", "industry"]]
    .drop_duplicates(["symbol", "yf_ticker"])
    .groupby("industry")
    .agg(
        stock_count=("symbol", "count"),
        symbols=("symbol", lambda x: ", ".join(sorted(x)))
    )
    .reset_index()
    .sort_values("stock_count", ascending=False)
)

display(industry_summary)

,industry,stock_count,symbols
8,Financial Services,23,"AXISBANK, BAJAJFINSV, BAJAJHLDNG, BAJFINANCE, ..."
0,Automobile and Auto Components,9,"BAJAJ-AUTO, BOSCHLTD, EICHERMOT, HYUNDAI, M&M,..."
1,Capital Goods,9,"ABB, BEL, CGPOWER, CUMMINSIND, ENRIN, HAL, MAZ..."
7,Fast Moving Consumer Goods,8,"BRITANNIA, GODREJCP, HINDUNILVR, ITC, NESTLEIN..."
9,Healthcare,8,"APOLLOHOSP, CIPLA, DIVISLAB, DRREDDY, MAXHEALT..."
11,Metals & Mining,7,"ADANIENT, HINDALCO, HINDZINC, JINDALSTEL, JSWS..."
10,Information Technology,6,"HCLTECH, INFY, LTM, TCS, TECHM, WIPRO"
12,Oil Gas & Consumable Fuels,6,"BPCL, COALINDIA, GAIL, IOC, ONGC, RELIANCE"
13,Power,6,"ADANIENSOL, ADANIGREEN, ADANIPOWER, NTPC, POWE..."
4,Construction Materials,4,"AMBUJACEM, GRASIM, SHREECEM, ULTRACEMCO"


In [24]:
display(INDEX_PREFIX_MAP) 

{'^NSEI': 'nifty50',
 '^CNX100': 'nifty100',
 '^INDIAVIX': 'indiavix',
 '^NSEBANK': 'nifty_bank',
 '^CNXAUTO': 'nifty_auto',
 '^CNXIT': 'nifty_it',
 '^CNXENERGY': 'nifty_energy',
 '^CNXFMCG': 'nifty_fmcg',
 '^CNXMETAL': 'nifty_metal',
 '^CNXPHARMA': 'nifty_pharma',
 '^CNXPSUBANK': 'nifty_psu_bank',
 '^CNXREALTY': 'nifty_realty',
 '^CNXCMDT': 'nifty_commodities',
 '^CNXMEDIA': 'nifty_media',
 '^CNXMNC': 'nifty_mnc',
 '^CNXPSE': 'nifty_pse'}

### Industry-to-sector proxy mapping

By looking at the available indices and the unique industries in our NIFTY 100 universe, we create a sector proxy mapping.

If we do not have a clean industry-specific index, we use **Nifty 100** as the fallback.

| Stock industry | Sector proxy index | Reason |
|---|---|---|
| Financial Services | Nifty Bank | Closest liquid financial-sector proxy |
| Automobile and Auto Components | Nifty Auto | Direct sector match |
| Capital Goods | Nifty 100 | No clean capital goods index in our selected v1 index set |
| Fast Moving Consumer Goods | Nifty FMCG | Direct sector match |
| Healthcare | Nifty Pharma | Closest healthcare/pharma proxy |
| Metals & Mining | Nifty Metal | Direct sector match |
| Information Technology | Nifty IT | Direct sector match |
| Oil Gas & Consumable Fuels | Nifty Energy | Closest energy/oil/gas proxy |
| Power | Nifty Energy | Closest energy-sector proxy |
| Construction Materials | Nifty Commodities | Closest available proxy |
| Consumer Services | Nifty 100 | No clean sector-specific proxy in v1 |
| Chemicals | Nifty Commodities | Closest available proxy |
| Consumer Durables | Nifty 100 | No clean sector-specific proxy in v1 |
| Services | Nifty 100 | No clean sector-specific proxy in v1 |
| Realty | Nifty Realty | Direct sector match |
| Construction | Nifty 100 | No clean construction index in v1 |
| Telecommunication | Nifty 100 | No telecom index in selected v1 set |

In [25]:
INDUSTRY_TO_SECTOR_INDEX = {
    "Financial Services": "^NSEBANK",
    "Automobile and Auto Components": "^CNXAUTO",
    "Capital Goods": "^CNX100",
    "Fast Moving Consumer Goods": "^CNXFMCG",
    "Healthcare": "^CNXPHARMA",
    "Metals & Mining": "^CNXMETAL",
    "Information Technology": "^CNXIT",
    "Oil Gas & Consumable Fuels": "^CNXENERGY",
    "Power": "^CNXENERGY",
    "Construction Materials": "^CNXCMDT",
    "Consumer Services": "^CNX100",
    "Chemicals": "^CNXCMDT",
    "Consumer Durables": "^CNX100",
    "Services": "^CNX100",
    "Realty": "^CNXREALTY",
    "Construction": "^CNX100",
    "Telecommunication": "^CNX100",
}

stock_features["sector_proxy_ticker"] = stock_features["industry"].map(INDUSTRY_TO_SECTOR_INDEX)

missing_sector_mapping = (
    stock_features[
        stock_features["sector_proxy_ticker"].isna()
    ][["industry"]]
    .drop_duplicates()
    .sort_values("industry")
)

print("Missing sector mappings:", len(missing_sector_mapping))
display(missing_sector_mapping)

stock_features["sector_proxy_name"] = stock_features["sector_proxy_ticker"].map(INDEX_PREFIX_MAP)

industry_sector_mapping_check = (
    stock_features[
        ["industry", "sector_proxy_ticker", "sector_proxy_name"]
    ]
    .drop_duplicates()
    .sort_values("industry")
)

display(industry_sector_mapping_check)

sector_proxy_stock_counts = (
    stock_features[
        ["symbol", "yf_ticker", "industry", "sector_proxy_ticker", "sector_proxy_name"]
    ]
    .drop_duplicates(["symbol", "yf_ticker"])
    .groupby(["sector_proxy_ticker", "sector_proxy_name"])
    .agg(
        stock_count=("symbol", "count"),
        industries=("industry", lambda x: ", ".join(sorted(set(x)))),
        symbols=("symbol", lambda x: ", ".join(sorted(x))),
    )
    .reset_index()
    .sort_values("stock_count", ascending=False)
)

display(sector_proxy_stock_counts)

Missing sector mappings: 0


,industry


,industry,sector_proxy_ticker,sector_proxy_name
37356,Automobile and Auto Components,^CNXAUTO,nifty_auto
0,Capital Goods,^CNX100,nifty100
242236,Chemicals,^CNXCMDT,nifty_commodities
200478,Construction,^CNX100,nifty100
21028,Construction Materials,^CNXCMDT,nifty_commodities
29192,Consumer Durables,^CNX100,nifty100
110619,Consumer Services,^CNX100,nifty100
74093,Fast Moving Consumer Goods,^CNXFMCG,nifty_fmcg
33274,Financial Services,^NSEBANK,nifty_bank
25110,Healthcare,^CNXPHARMA,nifty_pharma


,sector_proxy_ticker,sector_proxy_name,stock_count,industries,symbols
9,^NSEBANK,nifty_bank,23,Financial Services,"AXISBANK, BAJAJFINSV, BAJAJHLDNG, BAJFINANCE, ..."
0,^CNX100,nifty100,19,"Capital Goods, Construction, Consumer Durables...","ABB, ADANIPORTS, ASIANPAINT, BEL, BHARTIARTL, ..."
3,^CNXENERGY,nifty_energy,12,"Oil Gas & Consumable Fuels, Power","ADANIENSOL, ADANIGREEN, ADANIPOWER, BPCL, COAL..."
1,^CNXAUTO,nifty_auto,9,Automobile and Auto Components,"BAJAJ-AUTO, BOSCHLTD, EICHERMOT, HYUNDAI, M&M,..."
7,^CNXPHARMA,nifty_pharma,8,Healthcare,"APOLLOHOSP, CIPLA, DIVISLAB, DRREDDY, MAXHEALT..."
4,^CNXFMCG,nifty_fmcg,8,Fast Moving Consumer Goods,"BRITANNIA, GODREJCP, HINDUNILVR, ITC, NESTLEIN..."
6,^CNXMETAL,nifty_metal,7,Metals & Mining,"ADANIENT, HINDALCO, HINDZINC, JINDALSTEL, JSWS..."
2,^CNXCMDT,nifty_commodities,6,"Chemicals, Construction Materials","AMBUJACEM, GRASIM, PIDILITIND, SHREECEM, SOLAR..."
5,^CNXIT,nifty_it,6,Information Technology,"HCLTECH, INFY, LTM, TCS, TECHM, WIPRO"
8,^CNXREALTY,nifty_realty,2,Realty,"DLF, LODHA"


### Merge only matching sector index features

In [26]:
sector_feature_frames = []

sector_feature_cols = [
    "idx_close",
    "idx_return_1d",
    "idx_return_5d",
    "idx_return_20d",
    "idx_return_60d",
    "idx_daily_volatility_20d",
    "idx_daily_volatility_60d",
    "idx_annualized_volatility_20d",
    "idx_annualized_volatility_60d",
    "idx_close_to_ma_20",
    "idx_close_to_ma_50",
    "idx_close_to_ma_200",
]

for ticker, g in indices.groupby("yf_ticker"):
    # Create index features for this index
    gf = add_index_features(g)

    # Keep only needed columns
    keep_cols = ["date"] + [col for col in sector_feature_cols if col in gf.columns]
    gf = gf[keep_cols].copy()

    # Add ticker key for merging
    gf["sector_proxy_ticker"] = ticker

    # Rename index columns to generic sector columns
    rename_map = {
        "idx_close": "sector_close",
        "idx_return_1d": "sector_return_1d",
        "idx_return_5d": "sector_return_5d",
        "idx_return_20d": "sector_return_20d",
        "idx_return_60d": "sector_return_60d",
        "idx_daily_volatility_20d": "sector_daily_volatility_20d",
        "idx_daily_volatility_60d": "sector_daily_volatility_60d",
        "idx_annualized_volatility_20d": "sector_annualized_volatility_20d",
        "idx_annualized_volatility_60d": "sector_annualized_volatility_60d",
        "idx_close_to_ma_20": "sector_close_to_ma_20",
        "idx_close_to_ma_50": "sector_close_to_ma_50",
        "idx_close_to_ma_200": "sector_close_to_ma_200",
    }

    gf = gf.rename(columns=rename_map)

    sector_feature_frames.append(gf)

sector_features_long = pd.concat(sector_feature_frames, ignore_index=True)

sector_features_long = sector_features_long.sort_values(
    ["sector_proxy_ticker", "date"]
).reset_index(drop=True)

print("Sector features long shape:", sector_features_long.shape)

display(sector_features_long.head())
display(sector_features_long.tail())

Sector features long shape: (61691, 14)


,date,sector_close,sector_return_1d,sector_return_5d,sector_return_20d,sector_return_60d,sector_daily_volatility_20d,sector_daily_volatility_60d,sector_annualized_volatility_20d,sector_annualized_volatility_60d,sector_close_to_ma_20,sector_close_to_ma_50,sector_close_to_ma_200,sector_proxy_ticker
0,2010-01-04,5153.549805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,^CNX100
1,2010-01-05,5203.799805,0.009751,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,^CNX100
2,2010-01-06,5214.000000,0.001960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,^CNX100
3,2010-01-07,5191.700195,-0.004277,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,^CNX100
4,2010-01-08,5175.750000,-0.003072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,^CNX100


,date,sector_close,sector_return_1d,sector_return_5d,sector_return_20d,sector_return_60d,sector_daily_volatility_20d,sector_daily_volatility_60d,sector_annualized_volatility_20d,sector_annualized_volatility_60d,sector_close_to_ma_20,sector_close_to_ma_50,sector_close_to_ma_200,sector_proxy_ticker
61686,2026-07-08,23882.050781,-0.021175,-0.005157,0.027534,0.004498,0.008600,0.008324,0.136516,0.132141,-0.003023,0.002267,-0.039265,^NSEI
61687,2026-07-09,23962.800781,0.003381,-0.008806,0.032214,-0.003651,0.008588,0.008199,0.136338,0.130152,-0.001211,0.005766,-0.035794,^NSEI
61688,2026-07-10,24206.900391,0.010187,-0.002635,0.045131,0.015277,0.008741,0.008224,0.138758,0.130547,0.006770,0.015831,-0.025803,^NSEI
61689,2026-07-13,24211.000000,0.000169,-0.008979,0.024895,-0.000838,0.007692,0.007950,0.122111,0.126209,0.005710,0.015975,-0.025436,^NSEI
61690,2026-07-14,24052.050781,-0.006565,-0.014208,0.008307,-0.005980,0.007607,0.007994,0.120760,0.126897,-0.001303,0.009258,-0.031585,^NSEI


In [27]:
stock_features = stock_features.merge(
    sector_features_long,
    on=["date", "sector_proxy_ticker"],
    how="left"
)

print("Stock features shape after sector feature merge:", stock_features.shape)

important_sector_cols = [
    "sector_close",
    "sector_return_20d",
    "sector_return_60d",
    "sector_annualized_volatility_20d",
    "sector_close_to_ma_200",
]

missing_sector_cols = [
    col for col in important_sector_cols
    if col not in stock_features.columns
]

print("Missing sector columns:", missing_sector_cols)

display(
    stock_features[
        [
            "date",
            "symbol",
            "industry",
            "sector_proxy_ticker",
            "sector_proxy_name",
            "return_20d",
            "sector_return_20d",
            "sector_annualized_volatility_20d",
            "sector_close_to_ma_200",
        ]
    ]
    .dropna()
    .sample(30)
)

Stock features shape after sector feature merge: (357370, 123)
Missing sector columns: []


,date,symbol,industry,sector_proxy_ticker,sector_proxy_name,return_20d,sector_return_20d,sector_annualized_volatility_20d,sector_close_to_ma_200
75277,2014-10-17,BRITANNIA,Fast Moving Consumer Goods,^CNXFMCG,nifty_fmcg,-0.065026,-0.018257,0.168210,0.059886
346777,2016-09-16,VEDL,Metals & Mining,^CNXMETAL,nifty_metal,-0.052016,-0.057274,0.232760,0.217536
254383,2026-02-19,POWERGRID,Power,^CNXENERGY,nifty_energy,0.164808,0.081449,0.254812,0.019869
57084,2023-10-11,BANKBARODA,Financial Services,^NSEBANK,nifty_bank,0.021696,-0.023125,0.118813,0.031903
196176,2014-05-14,KOTAKBANK,Financial Services,^NSEBANK,nifty_bank,0.066778,0.097124,0.247815,0.286139
351105,2017-09-14,WIPRO,Information Technology,^CNXIT,nifty_it,-0.022418,-0.015352,0.122090,0.010309
226595,2012-10-09,NESTLEIND,Fast Moving Consumer Goods,^CNXFMCG,nifty_fmcg,0.006018,0.066958,0.197425,0.224930
209552,2020-04-01,M&M,Automobile and Auto Components,^CNXAUTO,nifty_auto,-0.414799,-0.333364,0.718120,-0.380249
162297,2011-12-02,ICICIBANK,Financial Services,^NSEBANK,nifty_bank,-0.109943,-0.062192,0.331210,-0.119505
348697,2024-06-28,VEDL,Metals & Mining,^CNXMETAL,nifty_metal,0.029946,0.028187,0.482995,0.229661


### Relative strength features (stock to nifty 50 , stock to sector)

In [28]:
# Relative return vs broad market:
# Positive value means the stock outperformed Nifty over that period.
# Negative value means the stock underperformed Nifty.
stock_features["relative_return_20d_vs_nifty50"] = (
    stock_features["return_20d"] - stock_features["nifty50_return_20d"]
)

stock_features["relative_return_60d_vs_nifty50"] = (
    stock_features["return_60d"] - stock_features["nifty50_return_60d"]
)

stock_features["relative_return_20d_vs_nifty100"] = (
    stock_features["return_20d"] - stock_features["nifty100_return_20d"]
)

stock_features["relative_return_60d_vs_nifty100"] = (
    stock_features["return_60d"] - stock_features["nifty100_return_60d"]
)

# Relative return vs sector:
# Positive value means the stock is stronger than its own sector proxy.
# Negative value means the stock is weaker than its sector.
stock_features["relative_return_20d_vs_sector"] = (
    stock_features["return_20d"] - stock_features["sector_return_20d"]
)

stock_features["relative_return_60d_vs_sector"] = (
    stock_features["return_60d"] - stock_features["sector_return_60d"]
)

# Volatility ratio vs market/sector:
# > 1 means stock is more volatile than benchmark.
# < 1 means stock is less volatile than benchmark.
stock_features["volatility_ratio_vs_nifty50_20d"] = (
    stock_features["annualized_volatility_20d"] / stock_features["nifty50_annualized_volatility_20d"]
)

stock_features["volatility_ratio_vs_nifty100_20d"] = (
    stock_features["annualized_volatility_20d"] / stock_features["nifty100_annualized_volatility_20d"]
)

stock_features["volatility_ratio_vs_sector_20d"] = (
    stock_features["annualized_volatility_20d"] / stock_features["sector_annualized_volatility_20d"]
)

print("Stock features shape:", stock_features.shape)

display(
    stock_features[
        [
            "date",
            "symbol",
            "industry",
            "sector_proxy_name",
            "return_20d",
            "nifty50_return_20d",
            "sector_return_20d",
            "relative_return_20d_vs_nifty50",
            "relative_return_20d_vs_sector",
            "annualized_volatility_20d",
            "sector_annualized_volatility_20d",
            "volatility_ratio_vs_sector_20d",
        ]
    ]
    .dropna()
    .sample(30, random_state=42)
)

Stock features shape: (357370, 132)


,date,symbol,industry,sector_proxy_name,return_20d,nifty50_return_20d,sector_return_20d,relative_return_20d_vs_nifty50,relative_return_20d_vs_sector,annualized_volatility_20d,sector_annualized_volatility_20d,volatility_ratio_vs_sector_20d
199867,2024-01-25,LODHA,Realty,nifty_realty,0.090904,-0.013953,0.076677,0.104858,0.014228,0.550222,0.362019,1.519868
167896,2018-01-30,INDHOTEL,Consumer Services,nifty100,0.211392,0.049280,0.048345,0.162112,0.163047,0.522858,0.074092,7.056886
31124,2017-11-02,ASIANPAINT,Consumer Durables,nifty100,0.023393,0.051327,0.055027,-0.027934,-0.031634,0.201859,0.077974,2.588784
37255,2026-02-17,AXISBANK,Financial Services,nifty_bank,0.038011,0.005468,0.021416,0.032543,0.016595,0.327638,0.156347,2.095580
335828,2015-02-13,UNIONBANK,Financial Services,nifty_bank,-0.241329,0.036655,0.006969,-0.277983,-0.248298,0.427580,0.238840,1.790236
295719,2018-09-05,TATAPOWER,Power,nifty_energy,0.042450,0.007891,0.010921,0.034559,0.031529,0.394248,0.163504,2.411241
3008,2022-03-11,ABB,Capital Goods,nifty100,-0.052748,-0.055402,-0.054094,0.002654,0.001346,0.311800,0.301659,1.033617
291701,2018-12-12,TATACONSUM,Fast Moving Consumer Goods,nifty_fmcg,-0.031825,0.014656,0.045236,-0.046482,-0.077062,0.338087,0.175143,1.930351
206501,2024-06-04,LTM,Information Technology,nifty_it,-0.015583,-0.024872,-0.027143,0.009289,0.011560,0.174439,0.144698,1.205537
286267,2014-03-19,SUNPHARMA,Healthcare,nifty_pharma,-0.032722,0.078649,0.004425,-0.111371,-0.037147,0.346770,0.184104,1.883554


### Market regime flags

These help the model understand whether the broad market is in a good/bad risk environment.

In [29]:
# Market trend regime:
# 1 means Nifty 50 is above its 200-day moving average.
# This usually indicates a healthier broad-market trend.
stock_features["nifty50_uptrend_flag"] = (
    stock_features["nifty50_close_to_ma_200"] > 0
).astype("Int64")

# Nifty 100 trend regime:
# Similar to Nifty 50, but broader large-cap market coverage.
stock_features["nifty100_uptrend_flag"] = (
    stock_features["nifty100_close_to_ma_200"] > 0
).astype("Int64")

# VIX risk regime:
# India VIX above 20 often indicates elevated fear/uncertainty.
stock_features["vix_above_20_flag"] = (
    stock_features["indiavix_close"] > 20
).astype("Int64")

# Higher stress regime:
# VIX above 25 marks a stronger risk-off environment.
stock_features["vix_above_25_flag"] = (
    stock_features["indiavix_close"] > 25
).astype("Int64")

# VIX 20-day change:
# Measures whether market fear/uncertainty has increased recently.
# VIX 20-day change:
# Measures whether market fear/uncertainty has increased recently.
# Calculated per stock row sequence to avoid ticker-boundary issues.
stock_features["vix_20d_change"] = (
    stock_features
    .sort_values(["yf_ticker", "date"])
    .groupby("yf_ticker")["indiavix_close"]
    .pct_change(20)
)

print("Stock features shape:", stock_features.shape)

display(
    stock_features[
        [
            "date",
            "symbol",
            "nifty50_close",
            "nifty50_close_to_ma_200",
            "nifty50_uptrend_flag",
            "nifty100_close_to_ma_200",
            "nifty100_uptrend_flag",
            "indiavix_close",
            "vix_above_20_flag",
            "vix_above_25_flag",
            "vix_20d_change",
        ]
    ]
    .dropna()
    .sample(30, random_state=42)
)

Stock features shape: (357370, 137)


,date,symbol,nifty50_close,nifty50_close_to_ma_200,nifty50_uptrend_flag,nifty100_close_to_ma_200,nifty100_uptrend_flag,indiavix_close,vix_above_20_flag,vix_above_25_flag,vix_20d_change
169451,2024-05-23,INDHOTEL,22967.650391,0.097257,1,0.133100,1,21.379999,1,0,1.096078
304249,2020-03-06,TCS,10989.450195,-0.059350,0,-0.057305,0,25.639999,1,1,0.860668
142189,2023-07-25,HDFCAMC,19680.599609,0.085441,1,0.079780,1,10.240000,0,0,-0.101754
285178,2026-05-11,SOLARINDS,23815.849609,-0.049884,0,-0.035922,0,18.549999,0,0,-0.015915
233250,2023-03-16,NTPC,16985.599609,-0.026563,0,-0.042746,0,16.219999,0,0,0.261275
263018,2019-04-05,SBILIFE,11665.950195,0.066043,1,0.058410,1,18.389999,0,0,0.178091
346538,2015-09-23,VEDL,7845.950195,-0.065002,0,-0.057051,0,20.690001,1,0,-0.229423
33051,2025-08-21,ASIANPAINT,25083.750000,0.042537,1,0.040896,1,11.370000,0,0,0.080798
178525,2017-05-25,IOC,9509.750000,0.091488,1,0.089638,1,10.450000,0,0,-0.109881
160287,2022-01-27,HINDZINC,17110.150391,0.029071,1,0.025794,1,21.070000,1,0,0.297414


### Feature readiness flags
These flags tell us which rows are usable for modelling after rolling windows and market/sector features are available.

In [30]:
# Base stock features needed for v1 modelling.
# These come only from the stock's own historical price/volume data.
base_required_features = [
    "return_20d",
    "return_60d",
    "annualized_volatility_20d",
    "annualized_volatility_60d",
    "atr_14_pct",
    "rsi_14",
    "adj_close_to_ma_50",
    "adj_close_to_ma_200",
    "drawdown_from_252d_high",
    "volume_ratio_20d",
]

# Global market features.
# These describe broad market condition and volatility.
market_required_features = [
    "nifty50_return_20d",
    "nifty50_return_60d",
    "nifty50_close_to_ma_200",
    "nifty100_return_20d",
    "nifty100_return_60d",
    "indiavix_close",
    "vix_20d_change",
]

# Sector proxy features.
# These describe the stock's mapped sector/index condition.
sector_required_features = [
    "sector_return_20d",
    "sector_return_60d",
    "sector_annualized_volatility_20d",
    "sector_close_to_ma_200",
]

required_model_features = (
    base_required_features
    + market_required_features
    + sector_required_features
)

missing_required_cols = [
    col for col in required_model_features
    if col not in stock_features.columns
]

print("Missing required feature columns:", missing_required_cols)

if missing_required_cols:
    raise ValueError(f"Missing required columns: {missing_required_cols}")

# Base readiness:
# 1 means stock-only features are available.
stock_features["base_features_ready"] = (
    stock_features[base_required_features].notna().all(axis=1)
).astype(int)

# Market readiness:
# 1 means stock + global market features are available.
stock_features["market_features_ready"] = (
    stock_features[base_required_features + market_required_features]
    .notna()
    .all(axis=1)
).astype(int)

# Sector readiness:
# 1 means stock + market + sector proxy features are available.
stock_features["sector_features_ready"] = (
    stock_features[required_model_features]
    .notna()
    .all(axis=1)
).astype(int)

# Main model-ready flag:
# Uses only research-universe stocks and rows where all required features exist.
# We do NOT remove manual-review large moves here; we only flag them.
stock_features["model_ready_v1"] = (
    (stock_features["is_research_universe"] == 1)
    & (stock_features["history_days_available"] >= 252)
    & (stock_features["sector_features_ready"] == 1)
).astype(int)

# Clean model-ready flag:
# Same as model_ready_v1, but excludes manual-review large-move rows.
# Useful later for A/B testing.
stock_features["model_ready_v1_clean"] = (
    (stock_features["model_ready_v1"] == 1)
    & (stock_features["manual_review_large_move_flag"] == 0)
).astype(int)

print("Stock features shape:", stock_features.shape)

print("\nBase features ready:")
display(stock_features["base_features_ready"].value_counts())

print("\nMarket features ready:")
display(stock_features["market_features_ready"].value_counts())

print("\nSector features ready:")
display(stock_features["sector_features_ready"].value_counts())

print("\nModel ready v1:")
display(stock_features["model_ready_v1"].value_counts())

print("\nModel ready v1 clean:")
display(stock_features["model_ready_v1_clean"].value_counts())

Missing required feature columns: []
Stock features shape: (357370, 142)

Base features ready:


base_features_ready
1    332416
0     24954
Name: count, dtype: int64


Market features ready:


market_features_ready
1    327230
0     30140
Name: count, dtype: int64


Sector features ready:


sector_features_ready
1    315165
0     42205
Name: count, dtype: int64


Model ready v1:


model_ready_v1
1    309073
0     48297
Name: count, dtype: int64


Model ready v1 clean:


model_ready_v1_clean
1    309038
0     48332
Name: count, dtype: int64

### Feature quality summary
This checks missing values column by column.

In [31]:
feature_quality = []

for col in stock_features.columns:
    feature_quality.append(
        {
            "column": col,
            "dtype": str(stock_features[col].dtype),
            "missing_count": stock_features[col].isna().sum(),
            "missing_pct": stock_features[col].isna().mean() * 100,
            "unique_count": stock_features[col].nunique(dropna=True),
        }
    )

feature_quality = (
    pd.DataFrame(feature_quality)
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

display(feature_quality.head(40))

feature_quality.to_csv(FEATURE_QUALITY_PATH, index=False)

print("Saved:", FEATURE_QUALITY_PATH)

,column,dtype,missing_count,missing_pct,unique_count
0,review_status,str,357319,99.985729,2
1,large_move_reason,str,357319,99.985729,2
2,sector_close_to_ma_200,float64,31703,8.871198,36544
3,rolling_high_252d,float64,24954,6.982679,24506
4,drawdown_from_252d_high,float64,24954,6.982679,303162
5,relative_return_60d_vs_sector,float64,21973,6.148530,335397
6,sector_daily_volatility_60d,float64,20722,5.798472,37934
7,sector_annualized_volatility_60d,float64,20722,5.798472,37934
8,sector_return_60d,float64,20722,5.798472,37930
9,adj_close_to_ma_200,float64,19858,5.556706,337512


Saved: E:\Projects\marketguard-india\data\interim\features\feature_quality_summary_v1.csv


### Sanity check latest snapshot
This checks the most recent date and confirms current dashboard rows look usable.

In [32]:
latest_date = stock_features["date"].max()

latest_snapshot = stock_features[
    stock_features["date"] == latest_date
].copy()

print("Latest date:", latest_date)
print("Latest snapshot rows:", len(latest_snapshot))
print("Unique stocks latest date:", latest_snapshot["yf_ticker"].nunique())

print("\nLatest snapshot model readiness:")
display(latest_snapshot["model_ready_v1"].value_counts(dropna=False))

print("\nLatest snapshot clean model readiness:")
display(latest_snapshot["model_ready_v1_clean"].value_counts(dropna=False))

print("\nLatest snapshot stock acceptance:")
display(latest_snapshot["stock_acceptance_status"].value_counts(dropna=False))

display(
    latest_snapshot[
        [
            "date",
            "symbol",
            "yf_ticker",
            "industry",
            "sector_proxy_name",
            "stock_acceptance_status",
            "model_ready_v1",
            "model_ready_v1_clean",
            "return_20d",
            "return_60d",
            "annualized_volatility_20d",
            "atr_14_pct",
            "rsi_14",
            "relative_return_20d_vs_nifty50",
            "relative_return_20d_vs_sector",
            "volatility_ratio_vs_sector_20d",
            "indiavix_close",
            "vix_above_20_flag",
            "nifty50_uptrend_flag",
        ]
    ]
    .sort_values("annualized_volatility_20d", ascending=False)
    .head(30)
)

Latest date: 2026-07-14 00:00:00
Latest snapshot rows: 100
Unique stocks latest date: 100

Latest snapshot model readiness:


model_ready_v1
1    90
0    10
Name: count, dtype: int64


Latest snapshot clean model readiness:


model_ready_v1_clean
1    90
0    10
Name: count, dtype: int64


Latest snapshot stock acceptance:


stock_acceptance_status
OK                    90
FLAG_SHORT_HISTORY    10
Name: count, dtype: int64

,date,symbol,yf_ticker,industry,sector_proxy_name,stock_acceptance_status,model_ready_v1,model_ready_v1_clean,return_20d,return_60d,annualized_volatility_20d,atr_14_pct,rsi_14,relative_return_20d_vs_nifty50,relative_return_20d_vs_sector,volatility_ratio_vs_sector_20d,indiavix_close,vix_above_20_flag,nifty50_uptrend_flag
326400,2026-07-14,TRENT,TRENT.NS,Consumer Services,nifty100,OK,1,1,-0.009904,-0.018333,0.553118,0.029534,37.934901,-0.018211,-0.017163,4.642474,13.75,0,0
121353,2026-07-14,ENRIN,ENRIN.NS,Capital Goods,nifty100,FLAG_SHORT_HISTORY,0,0,-0.060766,0.056543,0.456404,0.040108,43.928322,-0.069073,-0.068025,3.830730,13.75,0,0
176702,2026-07-14,INFY,INFY.NS,Information Technology,nifty_it,OK,1,1,-0.044334,-0.149749,0.442756,0.028698,52.226628,-0.052640,-0.067736,1.267564,13.75,0,0
140963,2026-07-14,HCLTECH,HCLTECH.NS,Information Technology,nifty_it,OK,1,1,0.006644,-0.174968,0.432018,0.035717,53.501939,-0.001663,-0.016759,1.236824,13.75,0,0
200477,2026-07-14,LODHA,LODHA.NS,Realty,nifty_realty,FLAG_SHORT_HISTORY,0,0,0.223146,0.310095,0.424229,0.040055,68.533430,0.214839,0.074905,1.464181,13.75,0,0
349205,2026-07-14,VEDL,VEDL.NS,Metals & Mining,nifty_metal,OK,1,1,-0.107518,-0.650887,0.383758,0.024361,30.662522,-0.115825,-0.076476,2.116082,13.75,0,0
207026,2026-07-14,LTM,LTM.NS,Information Technology,nifty_it,OK,1,1,0.014002,-0.134650,0.379136,0.032395,58.579282,0.005695,-0.009401,1.085428,13.75,0,0
305822,2026-07-14,TCS,TCS.NS,Information Technology,nifty_it,OK,1,1,0.000728,-0.145590,0.378916,0.029119,56.761962,-0.007579,-0.022675,1.084799,13.75,0,0
86338,2026-07-14,CGPOWER,CGPOWER.NS,Capital Goods,nifty100,OK,1,1,-0.028543,0.134865,0.373395,0.029158,48.451796,-0.036849,-0.035801,3.134011,13.75,0,0
110618,2026-07-14,DLF,DLF.NS,Realty,nifty_realty,OK,1,1,0.066979,0.105177,0.373148,0.030797,60.670763,0.058672,-0.081262,1.287882,13.75,0,0


### Feature dictionary
create a feature dictionary so the project is understandable later.

In [33]:
feature_dictionary_rows = [
    # Stock identity / metadata
    ("date", "Trading date"),
    ("symbol", "NSE stock symbol"),
    ("yf_ticker", "Yahoo Finance ticker"),
    ("company_name", "Company name"),
    ("industry", "NSE industry classification"),
    ("stock_acceptance_status", "Stock-level data quality status"),
    ("is_research_universe", "1 if stock is part of the 90-stock research/backtest universe"),
    ("is_limited_history", "1 if stock has limited trading history"),

    # Stock return / momentum
    ("return_1d", "1-day adjusted close return"),
    ("return_5d", "5-day adjusted close return"),
    ("return_20d", "20-day adjusted close return"),
    ("return_60d", "60-day adjusted close return"),
    ("return_120d", "120-day adjusted close return"),
    ("log_return_1d", "1-day log return using adjusted close"),

    # Stock volatility / risk
    ("daily_volatility_20d", "20-day rolling standard deviation of daily returns"),
    ("daily_volatility_60d", "60-day rolling standard deviation of daily returns"),
    ("annualized_volatility_20d", "20-day volatility annualized using 252 trading days"),
    ("annualized_volatility_60d", "60-day volatility annualized using 252 trading days"),
    ("true_range", "Daily true range using high, low, and previous close"),
    ("atr_14", "14-day average true range"),
    ("atr_14_pct", "14-day ATR divided by close price"),
    ("intraday_range_pct", "Daily high-low range divided by close price"),
    ("open_gap_pct", "Open price gap versus previous close"),
    ("close_position_in_day_range", "Close location within daily high-low range"),

    # Trend / drawdown
    ("ma_20", "20-day moving average of adjusted close"),
    ("ma_50", "50-day moving average of adjusted close"),
    ("ma_100", "100-day moving average of adjusted close"),
    ("ma_200", "200-day moving average of adjusted close"),
    ("adj_close_to_ma_20", "Adjusted close relative to 20-day moving average"),
    ("adj_close_to_ma_50", "Adjusted close relative to 50-day moving average"),
    ("adj_close_to_ma_100", "Adjusted close relative to 100-day moving average"),
    ("adj_close_to_ma_200", "Adjusted close relative to 200-day moving average"),
    ("drawdown_from_60d_high", "Drawdown from rolling 60-day high"),
    ("drawdown_from_252d_high", "Drawdown from rolling 252-day high"),
    ("rsi_14", "14-period RSI using adjusted close"),

    # Volume / liquidity
    ("volume_ratio_20d", "Current volume divided by 20-day average volume"),
    ("volume_ratio_60d", "Current volume divided by 60-day average volume"),
    ("turnover", "Close price multiplied by volume"),
    ("turnover_ratio_20d", "Current turnover divided by 20-day average turnover"),

    # Broad market features
    ("nifty50_return_20d", "20-day Nifty 50 return"),
    ("nifty50_return_60d", "60-day Nifty 50 return"),
    ("nifty100_return_20d", "20-day Nifty 100 return"),
    ("nifty100_return_60d", "60-day Nifty 100 return"),
    ("nifty50_close_to_ma_200", "Nifty 50 close relative to 200-day moving average"),
    ("nifty100_close_to_ma_200", "Nifty 100 close relative to 200-day moving average"),
    ("indiavix_close", "India VIX close value"),
    ("vix_20d_change", "20-day percentage change in India VIX"),

    # Sector proxy features
    ("sector_proxy_ticker", "Mapped sector index ticker for the stock"),
    ("sector_proxy_name", "Mapped sector index name"),
    ("sector_return_20d", "20-day return of mapped sector proxy"),
    ("sector_return_60d", "60-day return of mapped sector proxy"),
    ("sector_annualized_volatility_20d", "20-day annualized volatility of mapped sector proxy"),
    ("sector_close_to_ma_200", "Sector proxy close relative to 200-day moving average"),

    # Relative strength
    ("relative_return_20d_vs_nifty50", "Stock 20-day return minus Nifty 50 20-day return"),
    ("relative_return_60d_vs_nifty50", "Stock 60-day return minus Nifty 50 60-day return"),
    ("relative_return_20d_vs_nifty100", "Stock 20-day return minus Nifty 100 20-day return"),
    ("relative_return_60d_vs_nifty100", "Stock 60-day return minus Nifty 100 60-day return"),
    ("relative_return_20d_vs_sector", "Stock 20-day return minus sector proxy 20-day return"),
    ("relative_return_60d_vs_sector", "Stock 60-day return minus sector proxy 60-day return"),
    ("volatility_ratio_vs_nifty50_20d", "Stock 20-day volatility divided by Nifty 50 20-day volatility"),
    ("volatility_ratio_vs_nifty100_20d", "Stock 20-day volatility divided by Nifty 100 20-day volatility"),
    ("volatility_ratio_vs_sector_20d", "Stock 20-day volatility divided by sector proxy 20-day volatility"),

    # Regime / flags
    ("nifty50_uptrend_flag", "1 if Nifty 50 is above its 200-day moving average"),
    ("nifty100_uptrend_flag", "1 if Nifty 100 is above its 200-day moving average"),
    ("vix_above_20_flag", "1 if India VIX is above 20"),
    ("vix_above_25_flag", "1 if India VIX is above 25"),
    ("large_move_review_flag", "1 if row appears in large move review report"),
    ("manual_review_large_move_flag", "1 if row has large move requiring manual review"),
    ("keep_large_move_flag", "1 if large move is likely a real market event"),
    ("model_ready_v1", "1 if row is usable for v1 modelling"),
    ("model_ready_v1_clean", "1 if row is usable for clean v1 modelling excluding manual-review large moves"),
]

feature_dictionary = pd.DataFrame(
    feature_dictionary_rows,
    columns=["feature", "description"]
)

feature_dictionary["exists_in_dataset"] = feature_dictionary["feature"].isin(stock_features.columns)

display(feature_dictionary)

print("Missing dictionary features:")
display(feature_dictionary[feature_dictionary["exists_in_dataset"] == False])

feature_dictionary.to_csv(FEATURE_DICTIONARY_PATH, index=False)

print("Saved:", FEATURE_DICTIONARY_PATH)

,feature,description,exists_in_dataset
0,date,Trading date,True
1,symbol,NSE stock symbol,True
2,yf_ticker,Yahoo Finance ticker,True
3,company_name,Company name,True
4,industry,NSE industry classification,True
...,...,...,...
66,large_move_review_flag,1 if row appears in large move review report,True
67,manual_review_large_move_flag,1 if row has large move requiring manual review,True
68,keep_large_move_flag,1 if large move is likely a real market event,True
69,model_ready_v1,1 if row is usable for v1 modelling,True


Missing dictionary features:


,feature,description,exists_in_dataset


Saved: E:\Projects\marketguard-india\data\interim\features\feature_dictionary_v1.csv


In [34]:
# Sort final feature table for consistent downstream use
stock_features = (
    stock_features
    .sort_values(["yf_ticker", "date"])
    .reset_index(drop=True)
)

# Save CSV version
stock_features.to_csv(FEATURE_OUTPUT_CSV, index=False)

# Save Parquet version if parquet support is installed
parquet_saved = False

try:
    stock_features.to_parquet(FEATURE_OUTPUT_PARQUET, index=False)
    parquet_saved = True
except Exception as e:
    print("Parquet save skipped or failed.")
    print("Reason:", repr(e))

print("Final feature dataset shape:", stock_features.shape)
print("Saved CSV:", FEATURE_OUTPUT_CSV)

if parquet_saved:
    print("Saved Parquet:", FEATURE_OUTPUT_PARQUET)

print("\nModel-ready rows:")
display(stock_features["model_ready_v1"].value_counts())

print("\nClean model-ready rows:")
display(stock_features["model_ready_v1_clean"].value_counts())

print("\nLatest date:", stock_features["date"].max())
print("Unique stocks:", stock_features["yf_ticker"].nunique())

Final feature dataset shape: (357370, 142)
Saved CSV: E:\Projects\marketguard-india\data\interim\features\stock_features_v1.csv
Saved Parquet: E:\Projects\marketguard-india\data\interim\features\stock_features_v1.parquet

Model-ready rows:


model_ready_v1
1    309073
0     48297
Name: count, dtype: int64


Clean model-ready rows:


model_ready_v1_clean
1    309038
0     48332
Name: count, dtype: int64


Latest date: 2026-07-14 00:00:00
Unique stocks: 100


### Final feature table audit

In [35]:
# Final audit checks before closing feature engineering notebook

key_cols = ["yf_ticker", "date"]

duplicate_key_rows = stock_features.duplicated(key_cols).sum()

numeric_cols = stock_features.select_dtypes(include=["number"]).columns.tolist()

inf_summary = []

for col in numeric_cols:
    s = pd.to_numeric(stock_features[col], errors="coerce")
    inf_count = np.isinf(s).sum()
    
    if inf_count > 0:
        inf_summary.append(
            {
                "column": col,
                "inf_count": inf_count,
            }
        )

inf_summary = pd.DataFrame(inf_summary)

model_ready_missing_summary = (
    stock_features.loc[
        stock_features["model_ready_v1"] == 1,
        required_model_features
    ]
    .isna()
    .sum()
    .reset_index()
)

model_ready_missing_summary.columns = ["feature", "missing_count"]
model_ready_missing_summary = model_ready_missing_summary[
    model_ready_missing_summary["missing_count"] > 0
]

print("Final feature dataset shape:", stock_features.shape)
print("Duplicate yf_ticker-date rows:", duplicate_key_rows)
print("Numeric columns checked:", len(numeric_cols))
print("Columns with infinite values:", len(inf_summary))

print("\nModel-ready rows:", stock_features["model_ready_v1"].sum())
print("Clean model-ready rows:", stock_features["model_ready_v1_clean"].sum())
print("Latest date:", stock_features["date"].max())
print("Unique stocks:", stock_features["yf_ticker"].nunique())

print("\nInfinite value summary:")
display(inf_summary)

print("\nMissing required features inside model_ready_v1 rows:")
display(model_ready_missing_summary)

Final feature dataset shape: (357370, 142)
Duplicate yf_ticker-date rows: 0
Numeric columns checked: 127
Columns with infinite values: 0

Model-ready rows: 309073
Clean model-ready rows: 309038
Latest date: 2026-07-14 00:00:00
Unique stocks: 100

Infinite value summary:


""



Missing required features inside model_ready_v1 rows:


,feature,missing_count


## Feature Engineering Summary

This notebook created the first version of the MarketGuard India stock feature dataset.

### Final outputs

- `data/interim/features/stock_features_v1.csv`
- `data/interim/features/stock_features_v1.parquet`
- `data/interim/features/feature_quality_summary_v1.csv`
- `data/interim/features/feature_dictionary_v1.csv`

### Final dataset shape

- Rows: **357,370**
- Columns: **142**
- Unique stocks: **100**
- Latest date: **2026-07-14**

### Model-ready data

- `model_ready_v1`: **309,073 rows**
- `model_ready_v1_clean`: **309,038 rows**

### Validation checks

- Duplicate `yf_ticker` + `date` rows: **0**
- Infinite numeric values: **0**
- Missing required features inside `model_ready_v1` rows: **0**

### Notes

The dataset includes stock-level technical/risk features, broad-market features, India VIX regime features, sector proxy features, relative strength features, data-quality flags, and model-readiness flags.

Short-history stocks are retained for dashboard use but excluded from the v1 research/model-ready universe.